In [1]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Consistent style
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.edgecolor': '#333',
    'axes.labelcolor': '#333',
    'axes.titleweight': 'bold',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.color': '#555',
    'ytick.color': '#555',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
})

GREEN = '#2E7D32'
RED   = '#C62828'
BLUE  = '#1565C0'
GREY  = '#9E9E9E'

def fmt_k(x, _): 
    if abs(x) >= 1e6: return f'${x/1e6:.1f}M'
    if abs(x) >= 1e3: return f'${x/1e3:.0f}K'
    return f'${x:.0f}'

In [2]:
data = pd.read_csv(r"./merged_all.csv")

In [3]:
# ─── Setup ───────────────────────────────────────────────
data['report_date']  = pd.to_datetime(data['report_date'])
data['created_date'] = pd.to_datetime(data['created_date'])

# Unique site key — client + site_id
data['site_key']  = data['client_id'].astype(str) + '__' + data['site_id'].astype(str)
data['true_opex'] = data['cogs'] + data['expenses']

# US census region mapping
STATE_TO_REGION = {
    # Northeast
    'CT':'Northeast','ME':'Northeast','MA':'Northeast','NH':'Northeast','RI':'Northeast',
    'VT':'Northeast','NJ':'Northeast','NY':'Northeast','PA':'Northeast',
    # Midwest
    'IL':'Midwest','IN':'Midwest','MI':'Midwest','OH':'Midwest','WI':'Midwest',
    'IA':'Midwest','KS':'Midwest','MN':'Midwest','MO':'Midwest','NE':'Midwest',
    'ND':'Midwest','SD':'Midwest',
    # South
    'DE':'South','FL':'South','GA':'South','MD':'South','NC':'South','SC':'South',
    'VA':'South','WV':'South','DC':'South','AL':'South','KY':'South','MS':'South',
    'TN':'South','AR':'South','LA':'South','OK':'South','TX':'South',
    # West
    'AZ':'West','CO':'West','ID':'West','MT':'West','NV':'West','NM':'West',
    'UT':'West','WY':'West','AK':'West','CA':'West','HI':'West','OR':'West','WA':'West',
}
data['region'] = data['state'].map(STATE_TO_REGION)

# Site-level frame (for the map) and new-site cohort (for the time series)
site_pnl = data.groupby(
    ['site_key','client_id','client_name','site_id','location_name','city','state','region']
).agg(
    avg_monthly_revenue=('total_income','mean'),
    first_report=('report_date','min'),
    lat=('lat','first'),
    lon=('lon','first'),
).reset_index()

max_date = data['report_date'].max()
site_pnl['months_of_data'] = (
    (max_date.year  - site_pnl['first_report'].dt.year) * 12
  + (max_date.month - site_pnl['first_report'].dt.month) + 1
)
new_sites = site_pnl[(site_pnl['months_of_data'] >= 3) &
                     (site_pnl['months_of_data'] <= 30)].copy()

print(f"Total sites: {len(site_pnl)} · New cohort: {len(new_sites)}")
print("\nNew sites by region:")
print(new_sites['region'].value_counts())

Total sites: 162 · New cohort: 97

New sites by region:
region
South      68
Midwest    20
West        9
Name: count, dtype: int64


In [4]:
# ─── Robust state → region mapping (full name or 2-letter code) ───

REGION_BY_FULL_NAME = {
    # Midwest
    'Illinois':'Midwest','Indiana':'Midwest','Iowa':'Midwest','Kansas':'Midwest',
    'Michigan':'Midwest','Minnesota':'Midwest','Missouri':'Midwest','Nebraska':'Midwest',
    'North Dakota':'Midwest','Ohio':'Midwest','South Dakota':'Midwest','Wisconsin':'Midwest',
    # Northeast
    'Connecticut':'Northeast','Maine':'Northeast','Massachusetts':'Northeast',
    'New Hampshire':'Northeast','New Jersey':'Northeast','New York':'Northeast',
    'Pennsylvania':'Northeast','Rhode Island':'Northeast','Vermont':'Northeast',
    # South
    'Alabama':'South','Arkansas':'South','Delaware':'South','District of Columbia':'South',
    'Florida':'South','Georgia':'South','Kentucky':'South','Louisiana':'South',
    'Maryland':'South','Mississippi':'South','North Carolina':'South','Oklahoma':'South',
    'South Carolina':'South','Tennessee':'South','Texas':'South','Virginia':'South',
    'West Virginia':'South',
    # West
    'Alaska':'West','Arizona':'West','California':'West','Colorado':'West','Hawaii':'West',
    'Idaho':'West','Montana':'West','Nevada':'West','New Mexico':'West','Oregon':'West',
    'Utah':'West','Washington':'West','Wyoming':'West',
}

# Standard USPS 2-letter abbreviations → full name (for fallback)
ABBREV_TO_FULL = {
    'AL':'Alabama','AK':'Alaska','AZ':'Arizona','AR':'Arkansas','CA':'California',
    'CO':'Colorado','CT':'Connecticut','DE':'Delaware','DC':'District of Columbia',
    'FL':'Florida','GA':'Georgia','HI':'Hawaii','ID':'Idaho','IL':'Illinois',
    'IN':'Indiana','IA':'Iowa','KS':'Kansas','KY':'Kentucky','LA':'Louisiana',
    'ME':'Maine','MD':'Maryland','MA':'Massachusetts','MI':'Michigan','MN':'Minnesota',
    'MS':'Mississippi','MO':'Missouri','MT':'Montana','NE':'Nebraska','NV':'Nevada',
    'NH':'New Hampshire','NJ':'New Jersey','NM':'New Mexico','NY':'New York',
    'NC':'North Carolina','ND':'North Dakota','OH':'Ohio','OK':'Oklahoma','OR':'Oregon',
    'PA':'Pennsylvania','RI':'Rhode Island','SC':'South Carolina','SD':'South Dakota',
    'TN':'Tennessee','TX':'Texas','UT':'Utah','VT':'Vermont','VA':'Virginia',
    'WA':'Washington','WV':'West Virginia','WI':'Wisconsin','WY':'Wyoming',
}

def state_to_region(state_value):
    """Accept full name, 2-letter code, lowercase variants, or messy input."""
    if pd.isna(state_value):
        return None
    s = str(state_value).strip()
    # Try full name first (Title Case)
    if s.title() in REGION_BY_FULL_NAME:
        return REGION_BY_FULL_NAME[s.title()]
    # Try as 2-letter code
    s_upper = s.upper()
    if s_upper in ABBREV_TO_FULL:
        return REGION_BY_FULL_NAME[ABBREV_TO_FULL[s_upper]]
    return None

In [5]:
from utils import plot_site_unit_economics

In [6]:
def detect_opex_spikes(raw_data, threshold=1.2, min_trailing_months=4):
    """
    For each site, identify months where OPEX exceeds its 6-month trailing
    moving average by `threshold`× (e.g., 1.2 = 20% above baseline).
    """
    sub = raw_data.sort_values(['site_key','report_date']).copy()
    sub['true_opex'] = sub['cogs'] + sub['expenses']
    
    # 6-month trailing mean OPEX per site (excluding current month)
    sub['opex_baseline'] = (
        sub.groupby('site_key')['true_opex']
           .transform(lambda s: s.shift(1).rolling(6, min_periods=min_trailing_months).mean())
    )
    
    sub['opex_vs_baseline'] = sub['true_opex'] / sub['opex_baseline']
    sub['is_spike'] = sub['opex_vs_baseline'] > threshold
    
    spikes = sub[sub['is_spike']].copy()
    print(f"Spikes detected (OPEX > {threshold}× trailing 6mo baseline): {len(spikes)}")
    print(f"Unique sites with at least one spike: {spikes['site_key'].nunique()}")
    return sub, spikes

raw_with_spikes, spikes = detect_opex_spikes(data, threshold=1.2)

Spikes detected (OPEX > 1.2× trailing 6mo baseline): 264
Unique sites with at least one spike: 97


In [7]:
def build_spike_event_study(raw_with_spikes, spikes, pre_months=6, post_months=12):
    rd = raw_with_spikes.copy()
    rd['true_opex']    = rd['cogs'] + rd['expenses']
    rd['total_washes'] = rd['mem_wash_count'].fillna(0) + rd['ret_wash_count'].fillna(0)
    
    metrics = ['total_income', 'true_opex',
               'ASP_mem', 'ASP_ret',
               'total_washes',                       # ← new
               'mem_wash_count', 'ret_wash_count']
    
    records = []
    for _, spike in spikes.iterrows():
        site_ts = rd[rd['site_key'] == spike['site_key']].sort_values('report_date').copy()
        spike_date = spike['report_date']
        
        site_ts['months_from_spike'] = (
            (site_ts['report_date'].dt.year  - spike_date.year)  * 12 +
            (site_ts['report_date'].dt.month - spike_date.month)
        )
        window = site_ts[
            (site_ts['months_from_spike'] >= -pre_months) &
            (site_ts['months_from_spike'] <= post_months)
        ].copy()
        
        pre_mask = (window['months_from_spike'] >= -pre_months) & (window['months_from_spike'] < 0)
        baselines = {m: window.loc[pre_mask, m].mean() for m in metrics}
        
        if any(pd.isna(v) or v <= 0 for v in baselines.values()):
            continue
        
        for _, row in window.iterrows():
            rec = {
                'site_key':         spike['site_key'],
                'spike_date':       spike_date,
                'months_from_spike': int(row['months_from_spike']),
            }
            for m in metrics:
                if pd.notna(row[m]) and baselines[m] > 0:
                    rec[f'{m}_norm'] = row[m] / baselines[m]
                else:
                    rec[f'{m}_norm'] = np.nan
            records.append(rec)
    
    return pd.DataFrame(records)

# events_df = build_spike_event_study(raw_with_spikes, spikes)

events_df = build_spike_event_study(raw_with_spikes, spikes)
print(f"Event-study rows: {len(events_df)}")
print(f"Unique spike events: {events_df.groupby(['site_key','spike_date']).ngroups}")

Event-study rows: 4251
Unique spike events: 241


In [8]:
metrics_to_plot = [
    ('total_income_norm',   'Revenue (normalized)',                 '#2E7D32'),
    ('true_opex_norm',      'OPEX (normalized)',                    '#E65100'),
    ('ASP_mem_norm',        'ASP — Membership (normalized)',        '#1565C0'),
    ('ASP_ret_norm',        'ASP — Retail (normalized)',            '#D81B60'),
    ('total_washes_norm',   'Total Wash Count (normalized)',        '#5E35B1'),   # ← new
    ('mem_wash_count_norm', 'Wash count — Membership (normalized)', '#1565C0'),
    ('ret_wash_count_norm', 'Wash count — Retail (normalized)',     '#D81B60'),
]

fig = make_subplots(
    rows=len(metrics_to_plot), cols=1,
    shared_xaxes=True,
    subplot_titles=[m[1] for m in metrics_to_plot],
    vertical_spacing=0.05,
)

for i, (col, label, color) in enumerate(metrics_to_plot, start=1):
    agg = events_df.groupby('months_from_spike')[col].agg(
        median='median', q25=lambda s: s.quantile(0.25), q75=lambda s: s.quantile(0.75),
        n='count'
    ).reset_index()
    agg = agg[agg['n'] >= 5]   # need ≥5 events at each month
    
    x = agg['months_from_spike']
    
    # IQR band
    fig.add_trace(go.Scatter(
        x=list(x) + list(x[::-1]),
        y=list(agg['q75']) + list(agg['q25'][::-1]),
        fill='toself', fillcolor=color, opacity=0.12,
        line=dict(width=0), showlegend=False, hoverinfo='skip',
    ), row=i, col=1)
    
    # Median line
    fig.add_trace(go.Scatter(
        x=x, y=agg['median'],
        mode='lines+markers',
        line=dict(color=color, width=2.5), marker=dict(size=6),
        showlegend=False,
        hovertemplate=(f'<b>{label}</b><br>M%{{x}} from spike<br>'
                       'Median: %{y:.2f}× baseline<br>'
                       '<extra></extra>'),
    ), row=i, col=1)
    
    # Reference: 1.0 = no change from baseline
    fig.add_hline(y=1.0, line=dict(color='#333', dash='dash', width=1),
                  row=i, col=1)
    # Spike moment vertical line
    fig.add_vline(x=0, line=dict(color='#C62828', dash='dot', width=1.5),
                  row=i, col=1)

fig.update_xaxes(title_text='Months from OPEX spike', row=len(metrics_to_plot), col=1)
for i in range(1, len(metrics_to_plot)+1):
    fig.update_yaxes(title_text='× baseline', tickformat='.2f',
                      gridcolor='#EEE', row=i, col=1)

n_events = events_df.groupby(['site_key','spike_date']).ngroups
fig.update_layout(
    title=dict(
        text='<b>What happens around an OPEX spike?</b>'
             f'<br><sub>{n_events} spike events · '
             'dashed line = pre-spike baseline · red dotted = spike month · '
             'shaded band = 25-75th percentile</sub>',
        x=0.02, xanchor='left',
    ),
    height=240 * len(metrics_to_plot) + 100,
    plot_bgcolor='white', hovermode='x unified',
    margin=dict(l=70, r=40, t=110, b=60),
)
fig.show()

opex spikes are promotionmal discounts runs by clients, From this analysis we can genralize that when there is a OPEX spike , It leads to a revenue spike, Which leads to drop in Average selling Price of Membership(ASP), which leads to More conversions from retail customers to membes, which leads to More Members wash count in following months and less retail wash count in following month.

In [9]:
def build_distance_matrix(site_pnl_df, radius_km=20):
    """Vectorized haversine over all sites. Returns directed pairs within radius_km."""
    sites = (
        site_pnl_df.dropna(subset=['lat', 'lon'])[['site_key', 'lat', 'lon']]
        .drop_duplicates('site_key')
        .reset_index(drop=True)
    )
    lats = np.radians(sites['lat'].values)
    lons = np.radians(sites['lon'].values)

    dlat = lats[:, None] - lats[None, :]
    dlon = lons[:, None] - lons[None, :]
    a = np.sin(dlat / 2)**2 + np.cos(lats[:, None]) * np.cos(lats[None, :]) * np.sin(dlon / 2)**2
    dist_matrix = 2 * 6371.0 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    np.fill_diagonal(dist_matrix, np.inf)

    fi, ni = np.where(dist_matrix <= radius_km)
    dist_df = pd.DataFrame({
        'focal_key':    sites['site_key'].values[fi],
        'neighbor_key': sites['site_key'].values[ni],
        'distance_km':  dist_matrix[fi, ni],
    })
    print(f"Pairs within {radius_km}km: {len(dist_df)} | "
          f"Focal sites with ≥1 neighbor: {dist_df['focal_key'].nunique()}")
    return dist_df


In [10]:
def build_neighbor_event_study(raw_data, spikes, dist_df,
                                pre_months=6, post_months=12, min_pre_obs=3):
    """
    For each spike event at a focal site, collect neighbor metrics in the same
    [-pre, +post] window and normalize by the neighbor's own pre-spike baseline.
    """
    rd = raw_data.copy()
    rd['true_opex']    = rd['cogs'] + rd['expenses']
    rd['total_washes'] = rd['mem_wash_count'].fillna(0) + rd['ret_wash_count'].fillna(0)

    metrics = ['total_income', 'total_washes', 'mem_wash_count',
               'ret_wash_count', 'ASP_mem', 'ASP_ret']
    revenue_metrics = {'total_income'}  # must have positive baseline

    rd_by_site = {sk: grp.sort_values('report_date')
                  for sk, grp in rd.groupby('site_key')}

    nbr_lookup = (
        dist_df.groupby('focal_key')
               .apply(lambda g: list(zip(g['neighbor_key'], g['distance_km'])))
               .to_dict()
    )

    records = []
    isolated = 0

    for _, spike in spikes.iterrows():
        focal_key  = spike['site_key']
        spike_date = spike['report_date']
        nbrs = nbr_lookup.get(focal_key, [])
        if not nbrs:
            isolated += 1
            continue

        for nbr_key, dist_km in nbrs:
            if nbr_key not in rd_by_site:
                continue
            nbr_ts = rd_by_site[nbr_key].copy()
            nbr_ts['months_from_spike'] = (
                (nbr_ts['report_date'].dt.year  - spike_date.year)  * 12 +
                (nbr_ts['report_date'].dt.month - spike_date.month)
            )
            window = nbr_ts[
                (nbr_ts['months_from_spike'] >= -pre_months) &
                (nbr_ts['months_from_spike'] <= post_months)
            ].copy()

            pre_mask = (window['months_from_spike'] >= -pre_months) & (window['months_from_spike'] < 0)
            pre_data = window[pre_mask]
            if len(pre_data) < min_pre_obs:
                continue

            baselines, skip = {}, False
            for m in metrics:
                bv = pre_data[m].mean()
                if m in revenue_metrics and (pd.isna(bv) or bv <= 0):
                    skip = True
                    break
                baselines[m] = max(bv, 1) if (pd.isna(bv) or bv <= 0) else bv
            if skip:
                continue

            for _, row in window.iterrows():
                rec = {
                    'focal_key':          focal_key,
                    'spike_date':         spike_date,
                    'neighbor_key':       nbr_key,
                    'distance_km':        dist_km,
                    'months_from_spike':  int(row['months_from_spike']),
                }
                for m in metrics:
                    rv = row[m]
                    rec[f'{m}_raw']  = rv
                    rec[f'{m}_norm'] = (rv / baselines[m]
                                        if pd.notna(rv) and baselines[m] > 0
                                        else np.nan)
                records.append(rec)

    neighbor_events_df = pd.DataFrame(records)
    print(f"Isolated spike events (no neighbors within radius): {isolated}")
    print(f"Unique neighbor-event pairs: "
          f"{neighbor_events_df.groupby(['focal_key','spike_date','neighbor_key']).ngroups}")
    print(f"Total neighbor-event rows: {len(neighbor_events_df)}")
    return neighbor_events_df


In [11]:
def build_market_share_panel(raw_data, spikes, dist_df, pre_months=6, post_months=12):
    """
    For each spike event compute focal site's share of the local market
    (focal + all neighbors) at each month offset.
    """
    rd = raw_data.copy()
    rd['true_opex']    = rd['cogs'] + rd['expenses']
    rd['total_washes'] = rd['mem_wash_count'].fillna(0) + rd['ret_wash_count'].fillna(0)

    rd_by_site = {sk: grp.sort_values('report_date')
                  for sk, grp in rd.groupby('site_key')}

    nbr_lookup = (
        dist_df.groupby('focal_key')
               .apply(lambda g: list(zip(g['neighbor_key'], g['distance_km'])))
               .to_dict()
    )

    records = []
    for _, spike in spikes.iterrows():
        focal_key  = spike['site_key']
        spike_date = spike['report_date']
        nbrs = nbr_lookup.get(focal_key, [])
        if not nbrs:
            continue

        all_keys = [focal_key] + [sk for sk, _ in nbrs]
        participant_data = {}
        for sk in all_keys:
            if sk not in rd_by_site:
                continue
            ts = rd_by_site[sk].copy()
            ts['months_from_spike'] = (
                (ts['report_date'].dt.year  - spike_date.year)  * 12 +
                (ts['report_date'].dt.month - spike_date.month)
            )
            window = ts[
                (ts['months_from_spike'] >= -pre_months) &
                (ts['months_from_spike'] <= post_months)
            ].copy()
            if len(window) > 0:
                participant_data[sk] = window

        if focal_key not in participant_data:
            continue

        for mfs in sorted(participant_data[focal_key]['months_from_spike'].unique()):
            focal_row = participant_data[focal_key][
                participant_data[focal_key]['months_from_spike'] == mfs
            ]
            if len(focal_row) == 0:
                continue
            f_income = focal_row['total_income'].iloc[0]
            f_washes = focal_row['total_washes'].iloc[0]

            mkt_income = f_income
            mkt_washes = f_washes
            for sk, nbr_data in participant_data.items():
                if sk == focal_key:
                    continue
                nbr_row = nbr_data[nbr_data['months_from_spike'] == mfs]
                if len(nbr_row) > 0:
                    mkt_income += nbr_row['total_income'].fillna(0).iloc[0]
                    mkt_washes += nbr_row['total_washes'].fillna(0).iloc[0]

            records.append({
                'focal_key':           focal_key,
                'spike_date':          spike_date,
                'months_from_spike':   int(mfs),
                'focal_total_income':  f_income,
                'focal_total_washes':  f_washes,
                'market_total_income': mkt_income,
                'market_total_washes': mkt_washes,
                'focal_income_share':  f_income / mkt_income if mkt_income > 0 else np.nan,
                'focal_wash_share':    f_washes / mkt_washes if mkt_washes > 0 else np.nan,
            })

    market_share_df = pd.DataFrame(records)
    print(f"Market share events: "
          f"{market_share_df.groupby(['focal_key','spike_date']).ngroups}")
    return market_share_df


In [12]:
def compute_spillover_stats(events_df, neighbor_events_df, market_share_df, post_window=(1, 3)):
    """
    Concrete % numbers for the stealing vs market-growth hypothesis.
    post_window = (start_month, end_month) inclusive.
    """
    ndf = neighbor_events_df
    fdf = events_df
    mdf = market_share_df
    pw_start, pw_end = post_window

    post_mask_n = (ndf['months_from_spike'] >= pw_start) & (ndf['months_from_spike'] <= pw_end)
    post_mask_f = (fdf['months_from_spike'] >= pw_start) & (fdf['months_from_spike'] <= pw_end)

    def pair_median_nbr(col):
        return (
            ndf[post_mask_n]
            .groupby(['focal_key', 'spike_date', 'neighbor_key'])[col]
            .median().reset_index()
        )

    def pair_median_focal(col):
        return (
            fdf[post_mask_f]
            .groupby(['site_key', 'spike_date'])[col]
            .median().reset_index()
        )

    # Neighbor metrics
    post_ret   = pair_median_nbr('ret_wash_count_norm')
    post_mem   = pair_median_nbr('mem_wash_count_norm')
    post_total = pair_median_nbr('total_washes_norm')
    post_rev   = pair_median_nbr('total_income_norm')

    pct_nbr_ret   = (post_ret['ret_wash_count_norm']  - 1) * 100
    pct_nbr_mem   = (post_mem['mem_wash_count_norm']  - 1) * 100
    pct_nbr_total = (post_total['total_washes_norm']  - 1) * 100
    pct_nbr_rev   = (post_rev['total_income_norm']    - 1) * 100
    pct_decline   = (pct_nbr_rev < 0).mean() * 100

    # Focal metrics
    focal_ret   = pair_median_focal('ret_wash_count_norm')
    focal_mem   = pair_median_focal('mem_wash_count_norm')
    focal_total = pair_median_focal('total_washes_norm')
    focal_rev   = pair_median_focal('total_income_norm')

    pct_focal_ret   = (focal_ret['ret_wash_count_norm']  - 1) * 100
    pct_focal_mem   = (focal_mem['mem_wash_count_norm']  - 1) * 100
    pct_focal_total = (focal_total['total_washes_norm']  - 1) * 100
    pct_focal_rev   = (focal_rev['total_income_norm']    - 1) * 100

    # Market share
    pre_ms  = mdf[(mdf['months_from_spike'] >= -6) & (mdf['months_from_spike'] < 0)]
    post_ms = mdf[(mdf['months_from_spike'] >= pw_start) & (mdf['months_from_spike'] <= pw_end)]

    pre_share  = pre_ms.groupby(['focal_key','spike_date'])['focal_wash_share'].mean()
    post_share = post_ms.groupby(['focal_key','spike_date'])['focal_wash_share'].mean()
    share_df   = pd.DataFrame({'pre': pre_share, 'post': post_share}).dropna()

    stats = {
        'n_neighbor_event_pairs':             len(post_ret),
        # Neighbor
        'median_nbr_ret_wash_pct_change':     pct_nbr_ret.median(),
        'median_nbr_mem_wash_pct_change':     pct_nbr_mem.median(),
        'median_nbr_total_wash_pct_change':   pct_nbr_total.median(),
        'median_nbr_revenue_pct_change':      pct_nbr_rev.median(),
        'pct_nbr_sites_revenue_decline':      pct_decline,
        # Focal
        'median_focal_ret_wash_pct_change':   pct_focal_ret.median(),
        'median_focal_mem_wash_pct_change':   pct_focal_mem.median(),
        'median_focal_total_wash_pct_change': pct_focal_total.median(),
        'median_focal_revenue_pct_change':    pct_focal_rev.median(),
        # Market share
        'median_focal_wash_share_pre':        share_df['pre'].median()  * 100,
        'median_focal_wash_share_post':       share_df['post'].median() * 100,
        'share_gain_pp':                     (share_df['post'] - share_df['pre']).median() * 100,
        'n_market_share_events':              len(share_df),
    }

    w = f"M+{pw_start}–M+{pw_end}"
    print("=" * 66)
    print("  COMPETITIVE SPILLOVER ANALYSIS — KEY METRICS")
    print(f"  Post-window: {w}  |  {stats['n_neighbor_event_pairs']} neighbor-event pairs")
    print("=" * 66)
    print(f"  {'Metric':<38}  {'Focal':>8}  {'Neighbor':>8}")
    print(f"  {'-'*38}  {'-'*8}  {'-'*8}")
    print(f"  {'Retail wash count (' + w + ')':<38}  "
          f"{stats['median_focal_ret_wash_pct_change']:>+7.1f}%  "
          f"{stats['median_nbr_ret_wash_pct_change']:>+7.1f}%")
    print(f"  {'Membership wash count (' + w + ')':<38}  "
          f"{stats['median_focal_mem_wash_pct_change']:>+7.1f}%  "
          f"{stats['median_nbr_mem_wash_pct_change']:>+7.1f}%")
    print(f"  {'Total wash count (' + w + ')':<38}  "
          f"{stats['median_focal_total_wash_pct_change']:>+7.1f}%  "
          f"{stats['median_nbr_total_wash_pct_change']:>+7.1f}%")
    print(f"  {'Total revenue (' + w + ')':<38}  "
          f"{stats['median_focal_revenue_pct_change']:>+7.1f}%  "
          f"{stats['median_nbr_revenue_pct_change']:>+7.1f}%")
    print(f"  {'% sites with revenue decline':<38}  "
          f"{'—':>8}  "
          f"{stats['pct_nbr_sites_revenue_decline']:>7.0f}%")
    print()
    print(f"  Focal wash market share — pre-spike:   {stats['median_focal_wash_share_pre']:.1f}%")
    print(f"  Focal wash market share — post-spike:  {stats['median_focal_wash_share_post']:.1f}%")
    print(f"  Market share gain:                     {stats['share_gain_pp']:+.1f} pp")
    print(f"  (based on {stats['n_market_share_events']} spike events with ≥1 neighbor)")
    print("=" * 66)
    return stats


In [13]:
def plot_spillover_event_study(events_df, neighbor_events_df, market_share_df, stats,
                                min_events=5):
    """
    4-panel comparison: focal vs neighbor behavior around OPEX spikes.
    Panels 1-3: focal (blue) vs neighbor (orange), normalized to own baselines.
    Panel 4: focal site's share of local market wash volume (%).
    """
    FOCAL_COLOR = '#1565C0'   # blue
    NBR_COLOR   = '#E65100'   # orange
    SHARE_COLOR = '#6A1B9A'   # purple

    panels = [
        ('ret_wash_count_norm',  'Retail Wash Count (normalized to pre-spike baseline)'),
        ('total_income_norm',    'Revenue (normalized to pre-spike baseline)'),
        ('mem_wash_count_norm',  'Membership Wash Count (normalized to pre-spike baseline)'),
    ]

    fig = make_subplots(
        rows=4, cols=1,
        shared_xaxes=True,
        subplot_titles=[p[1] for p in panels] + ['Focal Site Market Share — Wash Volume (%)'],
        vertical_spacing=0.06,
    )

    for i, (col, title) in enumerate(panels, start=1):
        focal_agg = (
            events_df.groupby('months_from_spike')[col]
            .agg(median='median',
                 q25=lambda s: s.quantile(0.25),
                 q75=lambda s: s.quantile(0.75),
                 n='count')
            .reset_index()
        )
        focal_agg = focal_agg[focal_agg['n'] >= min_events]

        nbr_agg = (
            neighbor_events_df.groupby('months_from_spike')[col]
            .agg(median='median',
                 q25=lambda s: s.quantile(0.25),
                 q75=lambda s: s.quantile(0.75),
                 n='count')
            .reset_index()
        )
        nbr_agg = nbr_agg[nbr_agg['n'] >= min_events]

        for agg, color, label in [(focal_agg, FOCAL_COLOR, 'Focal site'),
                                   (nbr_agg,   NBR_COLOR,   'Neighbor sites')]:
            x = agg['months_from_spike']
            fig.add_trace(go.Scatter(
                x=list(x) + list(x[::-1]),
                y=list(agg['q75']) + list(agg['q25'][::-1]),
                fill='toself', fillcolor=color, opacity=0.10,
                line=dict(width=0), showlegend=False, hoverinfo='skip',
            ), row=i, col=1)
            fig.add_trace(go.Scatter(
                x=x, y=agg['median'],
                mode='lines+markers',
                line=dict(color=color, width=2.5),
                marker=dict(size=6),
                name=label, showlegend=(i == 1), legendgroup=label,
                hovertemplate=(f'<b>{label}</b><br>M%{{x}} from spike<br>'
                               'Median: %{y:.2f}× baseline<extra></extra>'),
            ), row=i, col=1)

        fig.add_hline(y=1.0, line=dict(color='#333', dash='dash', width=1), row=i, col=1)
        fig.add_vline(x=0,   line=dict(color='#C62828', dash='dot', width=1.5), row=i, col=1)
        fig.update_yaxes(title_text='× baseline', tickformat='.2f',
                         gridcolor='#EEE', row=i, col=1)

    # ── Panel 4: Market share ──
    ms_agg = (
        market_share_df.groupby('months_from_spike')['focal_wash_share']
        .agg(median='median',
             q25=lambda s: s.quantile(0.25),
             q75=lambda s: s.quantile(0.75),
             n='count')
        .reset_index()
    )
    ms_agg = ms_agg[ms_agg['n'] >= min_events]
    for col_name in ['median','q25','q75']:
        ms_agg[f'{col_name}_pct'] = ms_agg[col_name] * 100

    x = ms_agg['months_from_spike']
    fig.add_trace(go.Scatter(
        x=list(x) + list(x[::-1]),
        y=list(ms_agg['q75_pct']) + list(ms_agg['q25_pct'][::-1]),
        fill='toself', fillcolor=SHARE_COLOR, opacity=0.12,
        line=dict(width=0), showlegend=False, hoverinfo='skip',
    ), row=4, col=1)
    fig.add_trace(go.Scatter(
        x=x, y=ms_agg['median_pct'],
        mode='lines+markers',
        line=dict(color=SHARE_COLOR, width=2.5), marker=dict(size=6),
        name='Focal market share', showlegend=True, legendgroup='share',
        hovertemplate='M%{x}<br>Market share: %{y:.1f}%<extra></extra>',
    ), row=4, col=1)

    pre_s  = stats['median_focal_wash_share_pre']
    post_s = stats['median_focal_wash_share_post']
    fig.add_hline(y=pre_s, line=dict(color=SHARE_COLOR, dash='dash', width=1), row=4, col=1)
    fig.add_vline(x=0,     line=dict(color='#C62828',   dash='dot',  width=1.5), row=4, col=1)
    fig.add_annotation(x=-3, y=pre_s,  text=f'Pre: {pre_s:.1f}%',
                       showarrow=False, font=dict(size=11, color=SHARE_COLOR),
                       bgcolor='rgba(255,255,255,0.8)', row=4, col=1)
    fig.add_annotation(x=2,  y=post_s, text=f'Post: {post_s:.1f}%',
                       showarrow=False, font=dict(size=11, color=SHARE_COLOR),
                       bgcolor='rgba(255,255,255,0.8)', row=4, col=1)

    fig.update_yaxes(title_text='Market share (%)', ticksuffix='%',
                     gridcolor='#EEE', row=4, col=1)
    fig.update_xaxes(title_text='Months from OPEX spike', row=4, col=1,
                     showgrid=False, showline=True, linecolor='#CCC')
    for r in range(1, 4):
        fig.update_xaxes(showgrid=False, showline=True, linecolor='#CCC', row=r, col=1)

    ret_pct = stats['median_nbr_ret_wash_pct_change']
    rev_pct = stats['median_nbr_revenue_pct_change']
    sh_gain = stats['share_gain_pp']
    n_pairs = stats['n_neighbor_event_pairs']
    pct_dec = stats['pct_nbr_sites_revenue_decline']

    fig.update_layout(
        title=dict(
            text=(
                '<b>Does a focal OPEX spike steal from neighbors?</b>'
                f'<br><sub>'
                f'Neighbor retail washes M+1–M+3: <b>{ret_pct:+.1f}%</b>  ·  '
                f'Neighbor revenue: <b>{rev_pct:+.1f}%</b>  ·  '
                f'{pct_dec:.0f}% of neighbors see revenue decline  ·  '
                f'Focal wash share gain: <b>{sh_gain:+.1f} pp</b>  ·  '
                f'{n_pairs} neighbor-event pairs'
                f'</sub>'
            ),
            x=0.02, xanchor='left',
        ),
        height=300 * 4 + 160,
        plot_bgcolor='white', hovermode='x unified',
        legend=dict(orientation='h', yanchor='bottom', y=1.02,
                    xanchor='right', x=1, bgcolor='rgba(0,0,0,0)'),
        margin=dict(l=70, r=40, t=150, b=60),
    )
    fig.show()
    return fig


In [14]:
# ── Cell A: Distance matrix (run once) ──
dist_df = build_distance_matrix(site_pnl, radius_km=20)

# ── Cell B: Neighbor event study ──
neighbor_events_df = build_neighbor_event_study(
    data, spikes, dist_df,
    pre_months=6, post_months=12, min_pre_obs=3
)

# ── Cell C: Market share panel ──
market_share_df = build_market_share_panel(
    data, spikes, dist_df,
    pre_months=6, post_months=12
)

# ── Cell D: Concrete stats ──
stats = compute_spillover_stats(
    events_df, neighbor_events_df, market_share_df,
    post_window=(1, 3)
)

# ── Cell E: Visualization ──
fig = plot_spillover_event_study(events_df, neighbor_events_df, market_share_df, stats)


Pairs within 20km: 448 | Focal sites with ≥1 neighbor: 106


/var/folders/vl/swgny2hj4jlcvrx_1y6sxtrh0000gn/T/ipykernel_95634/1253833649.py:20: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: list(zip(g['neighbor_key'], g['distance_km'])))


Isolated spike events (no neighbors within radius): 89
Unique neighbor-event pairs: 388
Total neighbor-event rows: 6934


/var/folders/vl/swgny2hj4jlcvrx_1y6sxtrh0000gn/T/ipykernel_95634/1189689441.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: list(zip(g['neighbor_key'], g['distance_km'])))


Market share events: 168
  COMPETITIVE SPILLOVER ANALYSIS — KEY METRICS
  Post-window: M+1–M+3  |  359 neighbor-event pairs
  Metric                                     Focal  Neighbor
  --------------------------------------  --------  --------
  Retail wash count (M+1–M+3)                -7.0%    -13.6%
  Membership wash count (M+1–M+3)           +19.3%     +5.7%
  Total wash count (M+1–M+3)                +10.0%     -0.7%
  Total revenue (M+1–M+3)                   +16.7%     +4.7%
  % sites with revenue decline                   —       39%

  Focal wash market share — pre-spike:   28.4%
  Focal wash market share — post-spike:  30.3%
  Market share gain:                     +0.0 pp
  (based on 153 spike events with ≥1 neighbor)


In [15]:
# Geographic concentration check — are results dominated by one state/client?
focal_state = site_pnl.set_index('site_key')['state'].to_dict()
neighbor_events_df['focal_state'] = neighbor_events_df['focal_key'].map(focal_state)
print("Unique focal sites per state in neighbor event study:")
print(neighbor_events_df.groupby('focal_state')['focal_key'].nunique().sort_values(ascending=False))
print()
print("Neighbor-event pair count per state:")
print(neighbor_events_df.groupby('focal_state')[['focal_key']].count().rename(columns={'focal_key':'pairs'}).sort_values('pairs', ascending=False))


Unique focal sites per state in neighbor event study:
focal_state
TX    33
OH    10
MS     4
SC     3
GA     2
IN     2
TN     1
Name: focal_key, dtype: int64

Neighbor-event pair count per state:
             pairs
focal_state       
TX            5827
MS             697
OH             199
GA              73
SC              65
IN              46
TN              27


## Competitive Spillover Results
**359 Neighbor–Event Pairs | 20 km Radius**

| Metric | Value |Interpretation | 
|---------|---------|------:|
| Neighbor Retail Washes (M+1 to M+3) | **-13.6%** | the typical neighbor site washes 13.6% fewer retail to its pre spike baseline. |
| Neighbor Total Revenue (M+1 to M+3) | **+4.7%** | |
| Neighbor Sites with Revenue Decline | **39%** | roughly 39% of neighbors actually see revenue fall|
| Focal Site Market Share (Pre-Spike) | **28.4%** | |
| Focal Site Market Share (Post-Spike) | **30.3%** | |

### Interpretation

- **Strong evidence of customer switching:** Neighboring sites experienced a **13.6% decline in retail washes** following promotional spikes at focal sites.
- **Revenue remains resilient:** Despite lower retail traffic, neighboring sites saw a **4.7% increase in total revenue**, likely supported by stable membership revenue.
- **Impact is uneven:** **39% of neighboring sites experienced revenue declines**, suggesting lower-membership sites are more exposed to promotional cannibalization.
- **Market share shifted toward promoted sites:** Focal site share increased from **28.4% to 30.3%**, indicating that promotions captured volume from nearby competitors rather than solely generating incremental demand.

### Important Context

> **33 of the 55 focal sites with neighbors are located in the Texas RGV/BlueWave cluster.**

The observed spillover effects are therefore concentrated in a dense competitive market and may not fully generalize to the broader network.

A ~20% OPEX promotional spike causes neighbor retail washes to drop -13.6% in the following 3 months — strong evidence of customer stealing. The focal site captures some of those customers as members. However, the overall market share gain per event is small and inconsistent (median ≈ 0 pp), suggesting the stealing effect is real but concentrated in a subset of high-density markets (mostly TX/RGV in this data).

Steps:

1. we already have the events_df the precomputed one, we will use it directly
2. for each site we compute its distance from all neighbours within 20KM "dist_df".
3. then we compute the same stats like events_df, for this nearest neighbour from point 2. Here the main point is the neighbour is normalized by its own pre spike rolling averages.
4. then we compute the market_share_df, here we are computing gain in % share of the total market(when I am saying total market, It means within 20KM of radius, whatever sites are these that will be our market, so total washes here means withon 20KM total washes, and eventually we are computing % Wash Volume in pre and post spike.)
5. finally we are looking at the stats, all those numbers you are reporting above are median % changes like -13.6% that would be the median % change in retail wash count of neighbours, the +4.7% will be the median % chagne in neighborus total revenue, 39% are total neighbours whose revenue got down after the OPEX spike, a d the focal site market share is like in within 20KM radius out of total monthly wash column it was doing 28.4 after this promotional spike it was doing 30.3% of total wash volume. 

In [16]:
spikes.head(4)

,client_name,client_id,site_id,location_name,year,quarter,month,report_date,mem_purchase_count,mem_wash_count,...,location_id,source_schema,id,created_date,site_key,true_opex,region,opex_baseline,opex_vs_baseline,is_spike
5,Big Dan's,bigdans_000378,1,Big Dan's Rome,2023,2,6,2023-06-30,5181,16082,...,102,big_dans_car_wash,453,2021-09-15 14:12:43,bigdans_000378__1,245028.06,South,196061.332000,1.249752,True
218,Big Dan's,bigdans_000378,10,Big Dan's Tarpon Springs,2024,1,3,2024-03-31,3493,15582,...,205,big_dans_car_wash,462,2022-03-10 22:57:43,bigdans_000378__10,228922.76,South,166876.083333,1.371813,True
220,Big Dan's,bigdans_000378,10,Big Dan's Tarpon Springs,2024,2,5,2024-05-31,3513,14748,...,205,big_dans_car_wash,462,2022-03-10 22:57:43,bigdans_000378__10,217400.46,South,177583.050000,1.224219,True
244,Big Dan's,bigdans_000378,11,Big Dan's Fairburn,2024,1,3,2024-03-31,4701,13273,...,101,big_dans_car_wash,463,2022-05-16 13:33:29,bigdans_000378__11,242160.96,South,176069.900000,1.375368,True


## Section 1 — Campaign Spend & Duration
Group consecutive OPEX spike months per site into campaigns and quantify how much each campaign costs in incremental OPEX dollars and how long campaigns typically last.

In [17]:
def cluster_campaigns(spikes):
    """
    Group consecutive spike months into campaigns (gap <= 1 month = same campaign).
    Returns campaigns_df with spend, duration, and peak ratio per campaign.
    """
    records = []
    for site_key, grp in spikes.sort_values('report_date').groupby('site_key'):
        rows = grp.reset_index(drop=True)
        i = 0
        while i < len(rows):
            start_date = rows.loc[i, 'report_date']
            months  = [rows.loc[i, 'report_date']]
            incr    = [rows.loc[i, 'true_opex'] - rows.loc[i, 'opex_baseline']]
            ratios  = [rows.loc[i, 'opex_vs_baseline']]
            j = i + 1
            while j < len(rows):
                gap = (
                    (rows.loc[j,   'report_date'].year  - rows.loc[j-1, 'report_date'].year)  * 12 +
                    (rows.loc[j,   'report_date'].month - rows.loc[j-1, 'report_date'].month)
                )
                if gap <= 1:
                    months.append(rows.loc[j, 'report_date'])
                    incr.append(rows.loc[j, 'true_opex'] - rows.loc[j, 'opex_baseline'])
                    ratios.append(rows.loc[j, 'opex_vs_baseline'])
                    j += 1
                else:
                    break
            records.append({
                'site_key':               site_key,
                'campaign_start':         start_date,
                'campaign_end':           months[-1],
                'duration_months':        len(months),
                'total_incremental_opex': max(sum(incr), 0),
                'peak_opex_ratio':        max(ratios),
                'n_spikes':               len(months),
            })
            i = j

    campaigns_df = pd.DataFrame(records)
    tot = len(campaigns_df)

    def bucket_label(d):
        if d == 1:   return '1 month'
        if d <= 3:   return '2-3 months'
        return '4+ months'

    campaigns_df['duration_bucket'] = campaigns_df['duration_months'].apply(bucket_label)
    bucket_order = ['1 month', '2-3 months', '4+ months']

    def spend_row(df, label, tot):
        s = df['total_incremental_opex']
        n = len(df)
        return (f"  {label:<14} {n:>4}  ({n/tot*100:4.1f}%)  "
                f"mean ${s.mean():>8,.0f}  "
                f"std ${s.std():>8,.0f}  "
                f"p25 ${s.quantile(0.25):>8,.0f}  "
                f"p50 ${s.median():>8,.0f}  "
                f"p75 ${s.quantile(0.75):>8,.0f}")

    print(f"Raw spike months:            {len(spikes)}")
    print(f"Campaigns detected:          {tot}")
    print(f"Unique sites with campaigns: {campaigns_df['site_key'].nunique()}")
    print()
    hdr = (f"  {'Bucket':<14} {'N':>4}   {'%':>6}   "
           f"{'Mean':>12}  {'Std':>12}  {'p25':>12}  {'Median':>12}  {'p75':>12}")
    print(hdr)
    print("  " + "-" * (len(hdr) - 2))
    for b in bucket_order:
        sub = campaigns_df[campaigns_df['duration_bucket'] == b]
        if len(sub) == 0:
            continue
        s = sub['total_incremental_opex']
        n = len(sub)
        print(f"  {b:<14} {n:>4}  ({n/tot*100:4.1f}%)  "
              f"${s.mean():>10,.0f}  "
              f"${s.std():>10,.0f}  "
              f"${s.quantile(0.25):>10,.0f}  "
              f"${s.median():>10,.0f}  "
              f"${s.quantile(0.75):>10,.0f}")
    # All-campaigns row
    s_all = campaigns_df['total_incremental_opex']
    print(f"  {'All':<14} {tot:>4}  (100.0%)  "
          f"${s_all.mean():>10,.0f}  "
          f"${s_all.std():>10,.0f}  "
          f"${s_all.quantile(0.25):>10,.0f}  "
          f"${s_all.median():>10,.0f}  "
          f"${s_all.quantile(0.75):>10,.0f}")
    return campaigns_df

campaigns_df = cluster_campaigns(spikes)


Raw spike months:            264
Campaigns detected:          184
Unique sites with campaigns: 97

  Bucket            N        %           Mean           Std           p25        Median           p75
  ---------------------------------------------------------------------------------------------------
  1 month         141  (76.6%)  $    52,367  $    67,288  $    15,328  $    24,804  $    48,395
  2-3 months       35  (19.0%)  $    79,016  $    79,395  $    29,970  $    49,371  $    93,758
  4+ months         8  ( 4.3%)  $   196,937  $   148,733  $   127,879  $   167,029  $   211,681
  All             184  (100.0%)  $    63,722  $    80,006  $    16,770  $    29,894  $    69,532


In [18]:
campaigns_df

,site_key,campaign_start,campaign_end,duration_months,total_incremental_opex,peak_opex_ratio,n_spikes,duration_bucket
0,bigdans_000378__1,2023-06-30,2023-06-30,1,48966.728000,1.249752,1,1 month
1,bigdans_000378__10,2024-03-31,2024-03-31,1,62046.676667,1.371813,1,1 month
2,bigdans_000378__10,2024-05-31,2024-05-31,1,39817.410000,1.224219,1,1 month
3,bigdans_000378__11,2024-03-31,2024-03-31,1,66091.060000,1.375368,1,1 month
4,bigdans_000378__12,2024-05-31,2024-05-31,1,62374.500000,1.325618,1,1 month
...,...,...,...,...,...,...,...,...
179,vibecw_000482__2,2026-01-31,2026-02-28,2,72937.223333,1.493073,2,2-3 months
180,vibecw_000482__4,2024-12-31,2024-12-31,1,30582.255000,1.391992,1,1 month
181,vibecw_000482__5,2025-05-31,2025-06-30,2,41835.907333,1.320692,2,2-3 months
182,vibecw_000482__5,2026-01-31,2026-01-31,1,27464.740000,1.307637,1,1 month


In [19]:
campaigns_df['duration_bucket'] = pd.cut(
    campaigns_df['duration_months'],
    bins=[0, 1, 3, 100],
    labels=['1 month', '2-3 months', '4+ months']
)

color_map = {'1 month': '#1565C0', '2-3 months': '#E65100', '4+ months': '#6A1B9A'}

fig = go.Figure()
for bucket, color in color_map.items():
    sub = campaigns_df[campaigns_df['duration_bucket'] == bucket]
    if len(sub) == 0:
        continue
    fig.add_trace(go.Histogram(
        x=sub['total_incremental_opex'],
        name=bucket,
        marker_color=color,
        opacity=0.75,
        nbinsx=25,
    ))

med_spend = campaigns_df['total_incremental_opex'].median()
fig.add_vline(x=med_spend, line=dict(color='#333', dash='dash', width=1.5))
fig.add_annotation(
    x=med_spend, y=1, yref='paper',
    text=f'Median: ${med_spend:,.0f}',
    showarrow=True, arrowhead=2, ay=-30,
    font=dict(size=11), bgcolor='rgba(255,255,255,0.85)',
)
fig.update_layout(
    barmode='overlay',
    title=dict(
        text='<b>Campaign Incremental OPEX Spend Distribution</b>',
            #  '<br><sub>One entry per campaign · colored by duration</sub>',
        x=0.02, xanchor='left',
    ),
    xaxis_title='Incremental OPEX Spend ($)',
    yaxis_title='Number of Campaigns',
    plot_bgcolor='white',
    legend=dict(title='Campaign duration'),
    margin=dict(l=60, r=40, t=100, b=60),
    height=420,
)
fig.show()


In [20]:
# ── Multi-spike site distribution ────────────────────────────────────────────
spike_counts = spikes.groupby("site_key").size().rename("n_spikes")

# All sites in data, including those with zero spikes
all_sites    = data["site_key"].nunique()
sites_spiked = spike_counts.index.nunique()
sites_zero   = all_sites - sites_spiked

def bucket(n):
    if n == 1: return "1 spike"
    if n <= 3: return "2–3 spikes"
    if n <= 5: return "4–5 spikes"
    return "6+ spikes"

spike_counts_df = spike_counts.reset_index()
spike_counts_df["bucket"] = spike_counts_df["n_spikes"].apply(bucket)

bucket_order = ["1 spike", "2–3 spikes", "4–5 spikes", "6+ spikes"]
bucket_colors = {
    "1 spike":     "#90A4AE",
    "2–3 spikes":  "#1565C0",
    "4–5 spikes":  "#E65100",
    "6+ spikes":   "#6A1B9A",
}

print(f"Total sites in portfolio : {all_sites}")
print(f"Sites with ≥1 OPEX spike : {sites_spiked}  ({sites_spiked/all_sites*100:.1f}%)")
print(f"Sites with 0 spikes      : {sites_zero}  ({sites_zero/all_sites*100:.1f}%)")
print()
print(f"  {'Bucket':<14} {'Sites':>6}  {'%':>6}  {'Avg spikes':>10}  {'Max spikes':>10}")
print("  " + "-"*50)
for b in bucket_order:
    sub = spike_counts_df[spike_counts_df["bucket"] == b]
    if len(sub) == 0:
        continue
    n = len(sub)
    print(f"  {b:<14} {n:>6}  ({n/all_sites*100:4.1f}%)  "
          f"{sub['n_spikes'].mean():>9.1f}  {sub['n_spikes'].max():>10}")

print()
print(f"  {'Total spiking':<14} {sites_spiked:>6}  (100.0%)")

# ── Bar chart: sites by spike count ──────────────────────────────────────────
count_dist = spike_counts.value_counts().sort_index().reset_index()
count_dist.columns = ["n_spikes", "n_sites"]

bar_colors = [
    "#6A1B9A" if n >= 6 else
    "#E65100" if n >= 4 else
    "#1565C0" if n >= 2 else
    "#90A4AE"
    for n in count_dist["n_spikes"]
]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=count_dist["n_spikes"],
    y=count_dist["n_sites"],
    marker_color=bar_colors,
    hovertemplate="Sites with %{x} spike(s): %{y}<extra></extra>",
    text=count_dist["n_sites"],
    textposition="outside",
))

fig.update_layout(
    title=dict(
        text=(
            "<b>Sites by Number of OPEX Spikes</b>"
            f"<br><sub>{all_sites} total sites · {sites_zero} never spiked (grey = 0) · "
            "blue = 2–3 · orange = 4–5 · purple = 6+</sub>"
        ),
        x=0.02, xanchor="left",
    ),
    xaxis=dict(
        title="Number of OPEX Spikes per Site",
        dtick=1, showgrid=False, showline=True, linecolor="#CCC",
    ),
    yaxis=dict(title="Number of Sites", gridcolor="#EEE"),
    plot_bgcolor="white",
    height=400,
    margin=dict(l=70, r=40, t=110, b=60),
    bargap=0.25,
)

# add a zero-spike bar manually
fig.add_trace(go.Bar(
    x=[0], y=[sites_zero],
    marker_color="#CFD8DC",
    hovertemplate=f"Sites with 0 spikes: {sites_zero}<extra></extra>",
    text=[sites_zero], textposition="outside",
    showlegend=False,
))
fig.show()


Total sites in portfolio : 162
Sites with ≥1 OPEX spike : 97  (59.9%)
Sites with 0 spikes      : 65  (40.1%)

  Bucket          Sites       %  Avg spikes  Max spikes
  --------------------------------------------------
  1 spike            32  (19.8%)        1.0           1
  2–3 spikes         44  (27.2%)        2.5           3
  4–5 spikes         11  ( 6.8%)        4.4           5
  6+ spikes          10  ( 6.2%)        7.6           9

  Total spiking      97  (100.0%)


## Section 2 — Campaign Revenue ROI
For each campaign, compute monthly incremental revenue vs. the pre-campaign baseline, then track cumulative revenue recovery vs. the campaign spend. Answers: when does the median campaign break even?

In [21]:
def compute_campaign_roi(data, campaigns_df, pre_months=6, post_months=12):
    """
    For each campaign (anchored at campaign_start), compute incremental revenue
    vs. pre-campaign baseline and cumulative ROI = cumulative_incr / campaign_spend.
    Returns roi_df with one row per (campaign, month_offset).
    """
    rd = data.copy()
    rd['report_date'] = pd.to_datetime(rd['report_date'])
    rd_by_site = {sk: grp.sort_values('report_date') for sk, grp in rd.groupby('site_key')}

    base_records = []
    for _, campaign in campaigns_df.iterrows():
        site_key       = campaign['site_key']
        anchor_date    = campaign['campaign_start']
        campaign_spend = campaign['total_incremental_opex']
        if campaign_spend <= 0 or site_key not in rd_by_site:
            continue

        ts = rd_by_site[site_key].copy()
        ts['mfc'] = (
            (ts['report_date'].dt.year  - anchor_date.year)  * 12 +
            (ts['report_date'].dt.month - anchor_date.month)
        )
        window = ts[(ts['mfc'] >= -pre_months) & (ts['mfc'] <= post_months)]
        pre    = window[(window['mfc'] >= -pre_months) & (window['mfc'] < 0)]
        if len(pre) < 3:
            continue
        baseline_rev = pre['total_income'].mean()
        if pd.isna(baseline_rev) or baseline_rev <= 0:
            continue

        for _, row in window.iterrows():
            base_records.append({
                'site_key':             site_key,
                'campaign_start':       anchor_date,
                'months_from_campaign': int(row['mfc']),
                'incremental_revenue':  row['total_income'] - baseline_rev,
                'baseline_revenue':     baseline_rev,
                'campaign_spend':       campaign_spend,
            })

    if not base_records:
        print("No campaigns with sufficient revenue data.")
        return pd.DataFrame()

    base_df = pd.DataFrame(base_records)

    # Add cumulative ROI per campaign (running sum from M0 onwards)
    cum_records = []
    for (sk, cs), grp in base_df.groupby(['site_key', 'campaign_start']):
        grp   = grp.sort_values('months_from_campaign')
        spend = grp['campaign_spend'].iloc[0]
        cum_rev = 0.0
        for _, row in grp.iterrows():
            mfc = row['months_from_campaign']
            if mfc >= 0:
                cum_rev += row['incremental_revenue']
            cum_records.append({
                'site_key':                      sk,
                'campaign_start':                cs,
                'months_from_campaign':          mfc,
                'incremental_revenue':           row['incremental_revenue'],
                'baseline_revenue':              row['baseline_revenue'],
                'campaign_spend':                spend,
                'cumulative_incremental_revenue': cum_rev if mfc >= 0 else np.nan,
                'cumulative_roi':                (cum_rev / spend) if (mfc >= 0 and spend > 0) else np.nan,
            })

    roi_df = pd.DataFrame(cum_records)
    n_campaigns = roi_df.groupby(['site_key', 'campaign_start']).ngroups
    print(f"Campaigns with sufficient data for ROI: {n_campaigns}")
    return roi_df

roi_df = compute_campaign_roi(data, campaigns_df)


Campaigns with sufficient data for ROI: 175


In [22]:
if len(roi_df) == 0:
    print("No ROI data to plot.")
else:
    agg_incr = (
        roi_df.groupby('months_from_campaign')['incremental_revenue']
        .median().reset_index()
    )
    agg_roi = (
        roi_df[roi_df['months_from_campaign'] >= 0]
        .groupby('months_from_campaign')['cumulative_roi']
        .agg(median='median',
             q25=lambda s: s.quantile(0.25),
             q75=lambda s: s.quantile(0.75))
        .reset_index()
    )

    breakeven     = agg_roi[agg_roi['median'] >= 1.0]
    breakeven_month = int(breakeven['months_from_campaign'].iloc[0]) if len(breakeven) > 0 else None

    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        subplot_titles=[
            'Median Incremental Revenue per Month ($) vs. pre-campaign baseline',
            'Median Cumulative ROI (x campaign spend) — how long to break even?',
        ],
        vertical_spacing=0.14,
    )

    # Panel 1: incremental revenue bar chart
    bar_colors = ['#C62828' if x == 0 else ('#2E7D32' if x > 0 else '#9E9E9E')
                  for x in agg_incr['months_from_campaign']]
    fig.add_trace(go.Bar(
        x=agg_incr['months_from_campaign'],
        y=agg_incr['incremental_revenue'],
        marker_color=bar_colors,
        hovertemplate='M%{x}<br>Median incr. revenue: $%{y:,.0f}<extra></extra>',
    ), row=1, col=1)
    fig.add_hline(y=0, line=dict(color='#333', dash='dash', width=1), row=1, col=1)
    fig.add_vline(x=0, line=dict(color='#C62828', dash='dot', width=1.5), row=1, col=1)

    # Panel 2: cumulative ROI with IQR band
    x_roi = agg_roi['months_from_campaign']
    fig.add_trace(go.Scatter(
        x=list(x_roi) + list(x_roi[::-1]),
        y=list(agg_roi['q75']) + list(agg_roi['q25'][::-1]),
        fill='toself', fillcolor='#6A1B9A', opacity=0.12,
        line=dict(width=0), showlegend=False, hoverinfo='skip',
    ), row=2, col=1)
    fig.add_trace(go.Scatter(
        x=x_roi, y=agg_roi['median'],
        mode='lines+markers',
        line=dict(color='#6A1B9A', width=2.5), marker=dict(size=7),
        showlegend=False,
        hovertemplate='M+%{x}<br>Cumulative ROI: %{y:.2f}x<extra></extra>',
    ), row=2, col=1)
    fig.add_hline(y=1.0, line=dict(color='#2E7D32', dash='dash', width=1.5), row=2, col=1)
    fig.add_vline(x=0,   line=dict(color='#C62828', dash='dot',  width=1.5), row=2, col=1)
    if breakeven_month is not None:
        fig.add_vline(x=breakeven_month,
                      line=dict(color='#2E7D32', dash='dot', width=1.5), row=2, col=1)
        fig.add_annotation(
            x=breakeven_month + 0.3, y=agg_roi[agg_roi['months_from_campaign'] == breakeven_month]['median'].iloc[0],
            text=f'Break-even: M+{breakeven_month}',
            showarrow=False,
            font=dict(color='#2E7D32', size=11),
            bgcolor='rgba(255,255,255,0.85)',
            row=2, col=1,
        )

    fig.update_yaxes(title_text='Incremental Revenue ($)', tickprefix='$', gridcolor='#EEE', row=1, col=1)
    fig.update_yaxes(title_text='Cumulative ROI (x)', tickformat='.2f', gridcolor='#EEE', row=2, col=1)
    for r in [1, 2]:
        fig.update_xaxes(showgrid=False, showline=True, linecolor='#CCC', row=r, col=1)
    fig.update_xaxes(title_text='Months from Campaign Start', row=2, col=1)

    be_txt = (f'Median break-even: M+{breakeven_month}'
              if breakeven_month is not None else 'Break-even not reached within window')
    fig.update_layout(
        title=dict(
            text=f'<b>Campaign Revenue ROI</b><br><sub>{be_txt} · shaded band = 25–75th percentile</sub>',
            x=0.02, xanchor='left',
        ),
        showlegend=False,
        plot_bgcolor='white',
        height=640,
        margin=dict(l=80, r=40, t=110, b=60),
    )
    fig.show()


In [23]:
if len(roi_df) == 0:
    print("No ROI data to plot.")
else:
    # total_revenue_norm = actual / baseline  (1.0 = flat at baseline)
    roi_df['total_revenue_norm'] = (
        (roi_df['incremental_revenue'] + roi_df['baseline_revenue']) / roi_df['baseline_revenue']
    )

    agg_norm = (
        roi_df.groupby('months_from_campaign')['total_revenue_norm']
        .agg(median='median',
             q25=lambda s: s.quantile(0.25),
             q75=lambda s: s.quantile(0.75),
             n='count')
        .reset_index()
    )

    bar_colors = [
        '#C62828' if x == 0 else ('#2E7D32' if x > 0 else '#90A4AE')
        for x in agg_norm['months_from_campaign']
    ]

    fig = go.Figure()

    # IQR band
    x = agg_norm['months_from_campaign']
    fig.add_trace(go.Scatter(
        x=list(x) + list(x[::-1]),
        y=list(agg_norm['q75']) + list(agg_norm['q25'][::-1]),
        fill='toself', fillcolor='#1565C0', opacity=0.10,
        line=dict(width=0), showlegend=False, hoverinfo='skip',
    ))

    # Median bars
    fig.add_trace(go.Bar(
        x=x,
        y=agg_norm['median'],
        marker_color=bar_colors,
        hovertemplate='M%{x}<br>Median total revenue: %{y:.2f}× baseline<extra></extra>',
        name='Median total revenue (normalized)',
    ))

    fig.add_hline(y=1.0, line=dict(color='#333', dash='dash', width=1.5))
    fig.add_vline(x=0,   line=dict(color='#C62828', dash='dot', width=1.5))

    n_campaigns = roi_df.groupby(['site_key', 'campaign_start']).ngroups
    fig.update_layout(
        title=dict(
            text=('<b>Total Revenue MoM Around Promotional Spike</b>'
                  f'<br><sub>{n_campaigns} campaigns · 1.0 = pre-campaign baseline · ' 
                  'red dotted = campaign start · shaded = 25–75th percentile</sub>'),
            x=0.02, xanchor='left',
        ),
        xaxis_title='Months from Campaign Start',
        yaxis_title='Total Revenue (× baseline)',
        yaxis=dict(tickformat='.2f', gridcolor='#EEE'),
        xaxis=dict(showgrid=False, showline=True, linecolor='#CCC'),
        plot_bgcolor='white',
        showlegend=False,
        height=420,
        margin=dict(l=80, r=40, t=110, b=60),
    )
    fig.show()


In [24]:
def compute_total_revenue_by_spike(data, spikes, pre_months=6, post_months=12, min_pre_obs=3):
    """
    Anchors on every raw OPEX spike (all 264 spike-months).
    No clustering — each spike month is an independent event anchor.
    Returns a DataFrame with median total_income in $ at each month offset.
    Dropped only when a site has < min_pre_obs months before the spike.
    """
    rd = data.copy()
    rd['report_date'] = pd.to_datetime(rd['report_date'])
    rd_by_site = {sk: grp.sort_values('report_date') for sk, grp in rd.groupby('site_key')}

    records = []
    skipped = 0
    for _, spike in spikes.iterrows():
        site_key   = spike['site_key']
        spike_date = pd.to_datetime(spike['report_date'])
        if site_key not in rd_by_site:
            skipped += 1
            continue

        ts = rd_by_site[site_key].copy()
        ts['mfs'] = (
            (ts['report_date'].dt.year  - spike_date.year)  * 12 +
            (ts['report_date'].dt.month - spike_date.month)
        )
        window = ts[(ts['mfs'] >= -pre_months) & (ts['mfs'] <= post_months)]
        pre    = window[window['mfs'] < 0]
        if len(pre) < min_pre_obs:
            skipped += 1
            continue

        for _, row in window.iterrows():
            records.append({
                'site_key':             site_key,
                'spike_date':           spike_date,
                'months_from_spike':    int(row['mfs']),
                'total_income':         row['total_income'],
            })

    rev_spike_df = pd.DataFrame(records)
    n_spikes_used = rev_spike_df.groupby(['site_key', 'spike_date']).ngroups
    print(f"Total raw spikes    : {len(spikes)}")
    print(f"Spikes with data    : {n_spikes_used}  (dropped {skipped} — insufficient pre-spike history)")
    return rev_spike_df

rev_spike_df = compute_total_revenue_by_spike(data, spikes)


Total raw spikes    : 264
Spikes with data    : 253  (dropped 2 — insufficient pre-spike history)


In [25]:
if len(rev_spike_df) == 0:
    print("No spike revenue data to plot.")
else:
    agg = (
        rev_spike_df.groupby('months_from_spike')['total_income']
        .agg(median='median',
             q25=lambda s: s.quantile(0.25),
             q75=lambda s: s.quantile(0.75),
             n='count')
        .reset_index()
    )

    bar_colors = [
        '#C62828' if x == 0 else ('#2E7D32' if x > 0 else '#90A4AE')
        for x in agg['months_from_spike']
    ]

    fig = go.Figure()

    # IQR band
    x = agg['months_from_spike']
    fig.add_trace(go.Scatter(
        x=list(x) + list(x[::-1]),
        y=list(agg['q75']) + list(agg['q25'][::-1]),
        fill='toself', fillcolor='#1565C0', opacity=0.10,
        line=dict(width=0), showlegend=False, hoverinfo='skip',
    ))

    # Median bars
    fig.add_trace(go.Bar(
        x=x,
        y=agg['median'],
        marker_color=bar_colors,
        hovertemplate='M%{x}<br>Median total revenue: $%{y:,.0f}<extra></extra>',
    ))

    fig.add_vline(x=0, line=dict(color='#C62828', dash='dot', width=1.5))

    n_spikes_used = rev_spike_df.groupby(['site_key', 'spike_date']).ngroups
    fig.update_layout(
        title=dict(
            text=('<b>Total Revenue ($) by Month — All OPEX Spikes</b>'
                  f'<br><sub>{n_spikes_used} spike events · actual median $ · ' 
                  'red dotted = spike month · shaded = 25–75th percentile</sub>'),
            x=0.02, xanchor='left',
        ),
        xaxis_title='Months from OPEX Spike',
        yaxis_title='Median Total Revenue ($)',
        yaxis=dict(tickprefix='$', tickformat=',.0f', gridcolor='#EEE'),
        xaxis=dict(showgrid=False, showline=True, linecolor='#CCC'),
        plot_bgcolor='white',
        showlegend=False,
        height=420,
        margin=dict(l=90, r=40, t=110, b=60),
    )
    fig.show()


In [26]:
# ── OPEX / Revenue / Profit / Mem Purchases — 3 separate plots ───────────────
_rd = data.copy()
_rd["report_date"] = pd.to_datetime(_rd["report_date"])
_rd["true_opex"]   = _rd["cogs"] + _rd["expenses"]
_rd_by_site = {sk: grp.sort_values("report_date")
               for sk, grp in _rd.groupby("site_key")}

def reclassify(d):
    if d == 1: return "1 month"
    if d == 2: return "2 months"
    return "3+ months"

campaigns_df["dur_bucket2"] = campaigns_df["duration_months"].apply(reclassify)

_records = []
for _, camp in campaigns_df.iterrows():
    sk     = camp["site_key"]
    anchor = pd.to_datetime(camp["campaign_start"])
    bucket = camp["dur_bucket2"]
    dur    = int(camp["duration_months"])
    if sk not in _rd_by_site:
        continue
    ts = _rd_by_site[sk].copy()
    ts["mfs"] = (
        (ts["report_date"].dt.year  - anchor.year)  * 12 +
        (ts["report_date"].dt.month - anchor.month)
    )
    for _, row in ts[(ts["mfs"] >= -3) & (ts["mfs"] <= 6)].iterrows():
        _records.append({
            "bucket":           bucket,
            "duration":         dur,
            "mfs":              int(row["mfs"]),
            "opex":             row["true_opex"],
            "revenue":          row["total_income"],
            "profit":           row["total_income"] - row["true_opex"],
            "mem_purchases":    row["mem_purchase_count"],
        })

snap_df2 = pd.DataFrame(_records)

OPEX_COLOR    = "#4FC3F7"
REV_COLOR     = "#FFA726"
PROFIT_COLOR  = "#66BB6A"
MEM_COLOR     = "#CE93D8"   # purple — membership purchases
CAMP_SHADE    = "#FFF9C4"

BUCKET_CONFIG = {
    "1 month":   {"camp_months": [0],       "title": "1-Month Campaigns"},
    "2 months":  {"camp_months": [0, 1],    "title": "2-Month Campaigns"},
    "3+ months": {"camp_months": [0, 1, 2], "title": "3+ Month Campaigns"},
}

for bucket, cfg in BUCKET_CONFIG.items():
    sub = snap_df2[snap_df2["bucket"] == bucket]
    if len(sub) == 0:
        continue

    agg = (
        sub.groupby("mfs")[["opex", "revenue", "profit", "mem_purchases"]]
        .median()
        .reset_index()
    )
    n_camps     = campaigns_df[campaigns_df["dur_bucket2"] == bucket].shape[0]
    camp_months = cfg["camp_months"]
    month_list  = list(agg["mfs"])

    x_labels = []
    for m in month_list:
        if m in camp_months:
            x_labels.append(f"T={m}<br><b>📍 Campaign</b>")
        else:
            x_labels.append(f"T={m}")

    fig = go.Figure()

    # Shade campaign months
    for cm in camp_months:
        if cm in month_list:
            pos = month_list.index(cm)
            fig.add_vrect(
                x0=pos - 0.5, x1=pos + 0.5,
                fillcolor=CAMP_SHADE, opacity=0.55,
                layer="below", line_width=0,
            )

    fig.add_trace(go.Bar(
        name="OPEX", x=x_labels, y=agg["opex"],
        marker_color=OPEX_COLOR,
        hovertemplate="T=%{x}<br>OPEX: $%{y:,.0f}<extra></extra>",
    ))
    fig.add_trace(go.Bar(
        name="Revenue", x=x_labels, y=agg["revenue"],
        marker_color=REV_COLOR,
        hovertemplate="T=%{x}<br>Revenue: $%{y:,.0f}<extra></extra>",
    ))
    fig.add_trace(go.Bar(
        name="Profit (Rev − OPEX)", x=x_labels, y=agg["profit"],
        marker_color=PROFIT_COLOR,
        hovertemplate="T=%{x}<br>Profit: $%{y:,.0f}<extra></extra>",
    ))
    fig.add_trace(go.Bar(
        name="Membership Purchases", x=x_labels, y=agg["mem_purchases"],
        marker_color=MEM_COLOR,
        hovertemplate="T=%{x}<br>Mem Purchases: %{y:,.0f}<extra></extra>",
    ))

    camp_label = (
        "T=0 = campaign month"
        if len(camp_months) == 1
        else f"T={camp_months[0]}–T={camp_months[-1]} = campaign months (yellow)"
    )

    fig.update_layout(
        barmode="group",
        title=dict(
            text=(
                f"<b>OPEX · Revenue · Profit · Membership Purchases — {cfg['title']}</b>"
                f"<br><sub>n={n_camps} campaigns · {camp_label} · "
                "median values · T=-3 to T=-1 = pre-campaign baseline</sub>"
            ),
            x=0.02, xanchor="left",
        ),
        xaxis=dict(
            title="Month Offset from Campaign Start",
            showgrid=False, showline=True, linecolor="#CCC",
        ),
        yaxis=dict(
            title="Median Value",
            tickformat=",.0f",
            gridcolor="#EEE",
        ),
        legend=dict(orientation="h", x=0.02, y=1.01, xanchor="left"),
        plot_bgcolor="white",
        height=470,
        margin=dict(l=90, r=40, t=120, b=60),
        bargap=0.18,
        bargroupgap=0.04,
    )
    fig.show()


## Section 3 — Site Ramp Trajectory
Track OPEX spend, membership growth, and market share from a site's first month in the data. Aggregated across all 162 sites — shows the typical trajectory over the first 18 months.

In [27]:
def analyze_new_site_ramp(data, dist_df, max_age_months=18, min_months_data=6):
    """
    For each site, track OPEX, membership washes, and market share as a function
    of months since first appearance in data. Aggregate across all sites.
    """
    rd = data.copy()
    rd['report_date']  = pd.to_datetime(rd['report_date'])
    rd['true_opex']    = rd['cogs'] + rd['expenses']
    rd['total_washes'] = rd['mem_wash_count'].fillna(0) + rd['ret_wash_count'].fillna(0)

    # Fast lookup: date x site_key → total_washes
    washes_pivot = rd.pivot_table(
        index='report_date', columns='site_key', values='total_washes', aggfunc='sum'
    )

    first_reports = rd.groupby('site_key')['report_date'].min().to_dict()

    nbr_lookup = {}
    for focal, grp in dist_df.groupby('focal_key'):
        nbr_lookup[focal] = list(grp['neighbor_key'])

    rd_by_site = {sk: grp.sort_values('report_date') for sk, grp in rd.groupby('site_key')}

    records = []
    for site_key, ts in rd_by_site.items():
        first_date = first_reports[site_key]
        ts = ts.copy()
        ts['age_months'] = (
            (ts['report_date'].dt.year  - first_date.year)  * 12 +
            (ts['report_date'].dt.month - first_date.month) + 1
        )
        ts = ts[(ts['age_months'] >= 1) & (ts['age_months'] <= max_age_months)]
        if len(ts) < min_months_data:
            continue

        nbrs = nbr_lookup.get(site_key, [])

        for _, row in ts.iterrows():
            date     = row['report_date']
            f_washes = row['total_washes']

            mkt_share = np.nan
            if nbrs and date in washes_pivot.index:
                valid_nbrs = [n for n in nbrs if n in washes_pivot.columns]
                if valid_nbrs:
                    nbr_washes = washes_pivot.loc[date, valid_nbrs].fillna(0).sum()
                    mkt        = f_washes + nbr_washes
                    mkt_share  = f_washes / mkt if mkt > 0 else np.nan

            records.append({
                'site_key':       site_key,
                'age_months':     int(row['age_months']),
                'opex':           row['true_opex'],
                'mem_wash_count': row['mem_wash_count'],
                'total_washes':   f_washes,
                'market_share':   mkt_share,
                'has_neighbors':  len(nbrs) > 0,
            })

    ramp_df = pd.DataFrame(records)
    n_sites = ramp_df['site_key'].nunique()
    print(f"Sites in ramp analysis: {n_sites}")
    for m in [6, 12, 18]:
        ms = ramp_df[ramp_df['age_months'] == m]['market_share'].dropna()
        if len(ms) >= 5:
            print(f"  Median market share at month {m:2d}: {ms.median()*100:.1f}%  (n={len(ms)} sites with neighbors)")
    return ramp_df

ramp_df = analyze_new_site_ramp(data, dist_df)


Sites in ramp analysis: 130
  Median market share at month  6: 32.3%  (n=95 sites with neighbors)
  Median market share at month 12: 31.6%  (n=75 sites with neighbors)
  Median market share at month 18: 30.5%  (n=56 sites with neighbors)


In [28]:
RAMP_COLOR = '#1565C0'

fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    subplot_titles=[
        'OPEX ($) — median across all sites',
        'Membership Wash Count — median across all sites',
        'Market Share within 20 km (%) — sites with neighbors only',
    ],
    vertical_spacing=0.08,
)

specs = [
    ('opex',           'OPEX ($)',          False, ramp_df),
    ('mem_wash_count', 'Membership washes', False, ramp_df),
    ('market_share',   'Market share (%)',  True,  ramp_df[ramp_df['has_neighbors']]),
]

for i, (col, ylabel, is_pct, subset) in enumerate(specs, start=1):
    sub = subset.copy()
    if is_pct:
        sub[col] = sub[col] * 100

    agg = (
        sub.groupby('age_months')[col]
        .agg(median='median',
             q25=lambda s: s.quantile(0.25),
             q75=lambda s: s.quantile(0.75),
             n='count')
        .reset_index()
    )
    agg = agg[agg['n'] >= 5]
    x   = agg['age_months']

    fig.add_trace(go.Scatter(
        x=list(x) + list(x[::-1]),
        y=list(agg['q75']) + list(agg['q25'][::-1]),
        fill='toself', fillcolor=RAMP_COLOR, opacity=0.12,
        line=dict(width=0), showlegend=False, hoverinfo='skip',
    ), row=i, col=1)
    fig.add_trace(go.Scatter(
        x=x, y=agg['median'],
        mode='lines+markers',
        line=dict(color=RAMP_COLOR, width=2.5), marker=dict(size=6),
        showlegend=False,
        hovertemplate=f'Month %{{x}}<br>Median: %{{y:,.1f}}<extra></extra>',
    ), row=i, col=1)

    suffix = '%' if is_pct else ''
    fig.update_yaxes(title_text=ylabel, gridcolor='#EEE', ticksuffix=suffix, row=i, col=1)
    fig.update_xaxes(showgrid=False, showline=True, linecolor='#CCC', row=i, col=1)

n_sites = ramp_df['site_key'].nunique()
fig.update_xaxes(title_text='Months since first report', row=3, col=1)
fig.update_layout(
    title=dict(
        text=f'<b>Site Ramp Trajectory — First 18 Months in Data</b>'
             f'<br><sub>{n_sites} sites · shaded = 25–75th percentile</sub>',
        x=0.02, xanchor='left',
    ),
    plot_bgcolor='white',
    height=760,
    margin=dict(l=80, r=40, t=110, b=60),
)
fig.show()


## Section 4 — Incumbent Response to New Entrants
When a new site appears in the data, do established incumbents within 20 km react by changing their OPEX spend or losing retail volume? Incumbents are sites with ≥ 6 months of data before the new site's first report.

In [29]:
def analyze_incumbent_response(data, dist_df, pre_months=3, post_months=12,
                                min_pre_obs=3, incumbent_min_age=6):
    """
    For each site's first report month (treated as 'opening'), measure how
    established incumbents within 20 km respond in OPEX and retail washes.
    Incumbents must have >= incumbent_min_age months of data before the opening.
    """
    rd = data.copy()
    rd['report_date'] = pd.to_datetime(rd['report_date'])
    rd['true_opex']   = rd['cogs'] + rd['expenses']

    first_reports = rd.groupby('site_key')['report_date'].min().to_dict()
    rd_by_site    = {sk: grp.sort_values('report_date') for sk, grp in rd.groupby('site_key')}

    nbr_lookup = {}
    for focal, grp in dist_df.groupby('focal_key'):
        nbr_lookup[focal] = list(zip(grp['neighbor_key'], grp['distance_km']))

    records = []
    for new_site, opening_date in first_reports.items():
        neighbors = nbr_lookup.get(new_site, [])
        if not neighbors:
            continue

        for nbr_key, dist_km in neighbors:
            if nbr_key not in rd_by_site:
                continue
            nbr_first = first_reports[nbr_key]
            months_established = (
                (opening_date.year  - nbr_first.year)  * 12 +
                (opening_date.month - nbr_first.month)
            )
            if months_established < incumbent_min_age:
                continue

            nbr_ts = rd_by_site[nbr_key].copy()
            nbr_ts['mfs'] = (
                (nbr_ts['report_date'].dt.year  - opening_date.year)  * 12 +
                (nbr_ts['report_date'].dt.month - opening_date.month)
            )
            window = nbr_ts[(nbr_ts['mfs'] >= -pre_months) & (nbr_ts['mfs'] <= post_months)]
            pre    = window[window['mfs'] < 0]
            if len(pre) < min_pre_obs:
                continue

            baseline_opex = pre['true_opex'].mean()
            baseline_ret  = pre['ret_wash_count'].mean()
            if pd.isna(baseline_opex) or baseline_opex <= 0:
                continue

            for _, row in window.iterrows():
                records.append({
                    'new_site':            new_site,
                    'incumbent_key':       nbr_key,
                    'distance_km':         dist_km,
                    'months_from_opening': int(row['mfs']),
                    'opex_norm':           row['true_opex'] / baseline_opex,
                    'ret_wash_norm':       (row['ret_wash_count'] / max(baseline_ret, 1)
                                           if pd.notna(row['ret_wash_count']) else np.nan),
                })

    resp_df = pd.DataFrame(records)
    if len(resp_df) > 0:
        n_pairs = resp_df.groupby(['new_site', 'incumbent_key']).ngroups
        print(f"Incumbent-opening pairs: {n_pairs}")
    else:
        print("No incumbent-opening pairs found.")
    return resp_df

resp_df = analyze_incumbent_response(data, dist_df)


Incumbent-opening pairs: 113


In [30]:
if len(resp_df) == 0:
    print("No incumbent-opening pairs — cannot plot.")
else:
    INC_COLOR = '#C62828'

    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        subplot_titles=[
            'Incumbent OPEX (normalized)',
            'Incumbent Retail Wash Count (normalized)',
        ],
        vertical_spacing=0.10,
    )

    for i, col in enumerate(['opex_norm', 'ret_wash_norm'], start=1):
        agg = (
            resp_df.groupby('months_from_opening')[col]
            .agg(median='median',
                 q25=lambda s: s.quantile(0.25),
                 q75=lambda s: s.quantile(0.75),
                 n='count')
            .reset_index()
        )
        agg = agg[agg['n'] >= 5]
        x   = agg['months_from_opening']

        fig.add_trace(go.Scatter(
            x=list(x) + list(x[::-1]),
            y=list(agg['q75']) + list(agg['q25'][::-1]),
            fill='toself', fillcolor=INC_COLOR, opacity=0.12,
            line=dict(width=0), showlegend=False, hoverinfo='skip',
        ), row=i, col=1)
        fig.add_trace(go.Scatter(
            x=x, y=agg['median'],
            mode='lines+markers',
            line=dict(color=INC_COLOR, width=2.5), marker=dict(size=6),
            showlegend=False,
            hovertemplate='M%{x}<br>Median: %{y:.2f}× baseline<extra></extra>',
        ), row=i, col=1)

        fig.add_hline(y=1.0, line=dict(color='#333', dash='dash', width=1), row=i, col=1)
        fig.add_vline(x=0,   line=dict(color='#1565C0', dash='dot', width=1.5), row=i, col=1)
        fig.update_yaxes(title_text='× baseline', tickformat='.2f', gridcolor='#EEE', row=i, col=1)
        fig.update_xaxes(showgrid=False, showline=True, linecolor='#CCC', row=i, col=1)

    n_pairs = resp_df.groupby(['new_site', 'incumbent_key']).ngroups
    fig.update_xaxes(title_text='Months from new site opening', row=2, col=1)
    fig.update_layout(
        title=dict(
            text='<b>Incumbent Response to New Entrant Opening</b>'
                 f'<br><sub>{n_pairs} incumbent-opening pairs · '
                 'blue dotted = new site opens · dashed = pre-opening baseline · '
                 'shaded = 25–75th percentile</sub>',
            x=0.02, xanchor='left',
        ),
        plot_bgcolor='white',
        height=620,
        margin=dict(l=70, r=40, t=120, b=60),
    )
    fig.show()


In [31]:
def build_entrant_vs_incumbent_panel(data, dist_df,
                                      pre_months=6, post_months=12,
                                      entrant_norm_window=(1, 6),
                                      min_pre_obs=3, incumbent_min_age=6):
    rd = data.copy()
    rd["report_date"]  = pd.to_datetime(rd["report_date"])
    rd["true_opex"]    = rd["cogs"] + rd["expenses"]
    rd["total_washes"] = rd["mem_wash_count"].fillna(0) + rd["ret_wash_count"].fillna(0)

    METRICS = ["true_opex", "ASP_mem", "ASP_ret",
               "total_income", "mem_wash_count", "ret_wash_count", "total_washes"]

    # O(1) wash lookup: report_date × site_key
    washes_pivot = rd.pivot_table(
        index="report_date", columns="site_key", values="total_washes", aggfunc="sum"
    )

    first_reports = rd.groupby("site_key")["report_date"].min().to_dict()
    rd_by_site    = {sk: grp.sort_values("report_date").reset_index(drop=True)
                     for sk, grp in rd.groupby("site_key")}

    nbr_lookup = {}
    for focal, grp in dist_df.groupby("focal_key"):
        nbr_lookup[focal] = list(zip(grp["neighbor_key"], grp["distance_km"]))

    # All sites within 20km of each site (neighbors + self)
    market_sites = {}
    for focal, nbrs in nbr_lookup.items():
        market_sites[focal] = [focal] + [n for n, _ in nbrs]

    entrant_records = []
    incumb_records  = []

    for new_site, opening_date in first_reports.items():
        neighbors = nbr_lookup.get(new_site, [])
        if not neighbors or new_site not in rd_by_site:
            continue

        # ── New entrant ───────────────────────────────────────────────────
        ne_ts = rd_by_site[new_site].copy()
        ne_ts["mfo"] = (
            (ne_ts["report_date"].dt.year  - opening_date.year)  * 12 +
            (ne_ts["report_date"].dt.month - opening_date.month)
        )
        ne_window = ne_ts[(ne_ts["mfo"] >= 0) & (ne_ts["mfo"] <= post_months)]
        lo, hi = entrant_norm_window
        ne_base_rows = ne_ts[(ne_ts["mfo"] >= lo) & (ne_ts["mfo"] <= hi)]
        if len(ne_base_rows) < 3:
            continue

        all_market_sites = market_sites.get(new_site, [new_site])

        for _, row in ne_window.iterrows():
            rec = {"new_site": new_site, "months_from_opening": int(row["mfo"])}
            for m in METRICS:
                rec[m] = row[m]
            # market share: new site washes / total washes of all sites within 20km
            dt = row["report_date"]
            if dt in washes_pivot.index:
                row_washes = washes_pivot.loc[dt]
                mkt_total  = row_washes.reindex(all_market_sites).sum(min_count=1)
                ne_washes  = row_washes.get(new_site, np.nan)
                rec["market_share"] = (ne_washes / mkt_total * 100
                                       if pd.notna(mkt_total) and mkt_total > 0
                                       else np.nan)
            else:
                rec["market_share"] = np.nan
            entrant_records.append(rec)

        # ── Incumbents ────────────────────────────────────────────────────
        for nbr_key, dist_km in neighbors:
            if nbr_key not in rd_by_site:
                continue
            nbr_first = first_reports.get(nbr_key)
            if nbr_first is None:
                continue
            months_estab = (
                (opening_date.year  - nbr_first.year)  * 12 +
                (opening_date.month - nbr_first.month)
            )
            if months_estab < incumbent_min_age:
                continue

            nbr_ts = rd_by_site[nbr_key].copy()
            nbr_ts["mfo"] = (
                (nbr_ts["report_date"].dt.year  - opening_date.year)  * 12 +
                (nbr_ts["report_date"].dt.month - opening_date.month)
            )
            window = nbr_ts[
                (nbr_ts["mfo"] >= -pre_months) & (nbr_ts["mfo"] <= post_months)
            ]
            pre = window[window["mfo"] < 0]
            if len(pre) < min_pre_obs:
                continue
            if pd.isna(pre["total_income"].mean()) or pre["total_income"].mean() <= 0:
                continue

            for _, row in window.iterrows():
                rec = {
                    "new_site":            new_site,
                    "incumbent_key":       nbr_key,
                    "distance_km":         dist_km,
                    "months_from_opening": int(row["mfo"]),
                }
                for m in METRICS:
                    rec[m] = row[m]
                incumb_records.append(rec)

    entrant_df = pd.DataFrame(entrant_records)
    incumb_df  = pd.DataFrame(incumb_records)
    n_pairs    = incumb_df.groupby(["new_site", "incumbent_key"]).ngroups if len(incumb_df) else 0
    print(f"New sites tracked      : {entrant_df['new_site'].nunique() if len(entrant_df) else 0}")
    print(f"Incumbent-site pairs   : {n_pairs}")
    return entrant_df, incumb_df

entrant_df, incumb_df = build_entrant_vs_incumbent_panel(data, dist_df)


New sites tracked      : 92
Incumbent-site pairs   : 59


In [32]:
if len(entrant_df) == 0 or len(incumb_df) == 0:
    print("Insufficient data for entrant vs incumbent panel.")
else:
    METRICS_META = [
        ("true_opex",      "OPEX ($)",               "$"),
        ("ASP_mem",        "ASP — Membership ($)",   "$"),
        ("ASP_ret",        "ASP — Retail ($)",       "$"),
        ("total_income",   "Total Revenue ($)",      "$"),
        ("mem_wash_count", "Membership Wash Count",  ""),
        ("ret_wash_count", "Retail Wash Count",      ""),
        ("total_washes",   "Total Wash Count",       ""),
    ]

    NE_COLOR  = "#1565C0"
    INC_COLOR = "#E65100"

    fig = make_subplots(
        rows=8, cols=1,
        shared_xaxes=True,
        subplot_titles=[m[1] for m in METRICS_META] + ["New Entrant Market Share (%)"],
        vertical_spacing=0.04,
    )

    # Panels 1-7: entrant vs incumbent raw metrics
    for row_idx, (col, label, prefix) in enumerate(METRICS_META, start=1):

        for df, color, series_name in [
            (entrant_df, NE_COLOR,  "New Entrant"),
            (incumb_df,  INC_COLOR, "Incumbent"),
        ]:
            grp_col = "months_from_opening"
            agg = (
                df.groupby(grp_col)[col]
                .agg(median="median",
                     q25=lambda s: s.quantile(0.25),
                     q75=lambda s: s.quantile(0.75),
                     n="count")
                .reset_index()
            )
            agg = agg[agg["n"] >= 5]
            x = agg[grp_col]

            fig.add_trace(go.Scatter(
                x=list(x) + list(x[::-1]),
                y=list(agg["q75"]) + list(agg["q25"][::-1]),
                fill="toself", fillcolor=color, opacity=0.10,
                line=dict(width=0), showlegend=False, hoverinfo="skip",
            ), row=row_idx, col=1)
            fig.add_trace(go.Scatter(
                x=x, y=agg["median"],
                mode="lines+markers",
                name=series_name if row_idx == 1 else None,
                legendgroup=series_name, showlegend=(row_idx == 1),
                line=dict(color=color, width=2.5), marker=dict(size=5),
                hovertemplate=(
                    f"M%{{x}}<br>{label}: {prefix}%{{y:,.0f}}<extra>{series_name}</extra>"
                    if prefix else
                    f"M%{{x}}<br>{label}: %{{y:,.0f}}<extra>{series_name}</extra>"
                ),
            ), row=row_idx, col=1)

        fig.add_vline(x=0, line=dict(color="#1B5E20", dash="dot", width=1.5),
                      row=row_idx, col=1)
        y_fmt = "$.3s" if prefix == "$" else ","
        fig.update_yaxes(title_text=label, tickformat=y_fmt,
                         gridcolor="#EEE", row=row_idx, col=1)
        fig.update_xaxes(showgrid=False, showline=True, linecolor="#CCC",
                         row=row_idx, col=1)

    # Panel 8: market share (new entrant only — incumbents have no single share)
    agg_ms = (
        entrant_df.groupby("months_from_opening")["market_share"]
        .agg(median="median",
             q25=lambda s: s.quantile(0.25),
             q75=lambda s: s.quantile(0.75),
             n="count")
        .reset_index()
    )
    agg_ms = agg_ms[agg_ms["n"] >= 5]
    x_ms = agg_ms["months_from_opening"]

    fig.add_trace(go.Scatter(
        x=list(x_ms) + list(x_ms[::-1]),
        y=list(agg_ms["q75"]) + list(agg_ms["q25"][::-1]),
        fill="toself", fillcolor=NE_COLOR, opacity=0.12,
        line=dict(width=0), showlegend=False, hoverinfo="skip",
    ), row=8, col=1)
    fig.add_trace(go.Scatter(
        x=x_ms, y=agg_ms["median"],
        mode="lines+markers",
        name="Market Share" if True else None,
        legendgroup="ms", showlegend=True,
        line=dict(color=NE_COLOR, width=2.5), marker=dict(size=5),
        hovertemplate="M%{x}<br>Market share: %{y:.1f}%<extra>New Entrant</extra>",
    ), row=8, col=1)

    fig.add_vline(x=0, line=dict(color="#1B5E20", dash="dot", width=1.5),
                  row=8, col=1)
    fig.update_yaxes(title_text="Market Share (%)", ticksuffix="%",
                     gridcolor="#EEE", row=8, col=1)
    fig.update_xaxes(showgrid=False, showline=True, linecolor="#CCC",
                     title_text="Months from New Entrant Opening", row=8, col=1)

    n_sites = entrant_df["new_site"].nunique()
    n_pairs = incumb_df.groupby(["new_site", "incumbent_key"]).ngroups
    fig.update_layout(
        title=dict(
            text=(
                "<b>New Entrant vs Incumbent — 8 Metrics Around Opening</b>"
                f"<br><sub>{n_sites} new sites · {n_pairs} incumbent pairs · "
                "blue = new entrant · orange = incumbent · "
                "shaded = 25–75th percentile · green dotted = opening month</sub>"
            ),
            x=0.02, xanchor="left",
        ),
        legend=dict(orientation="h", x=0.02, y=1.012, xanchor="left"),
        plot_bgcolor="white",
        height=1900,
        margin=dict(l=110, r=40, t=130, b=60),
    )
    fig.show()


## Section 5 — Neighbor Response to Promotional Spikes
Same structure as Section 4 but the event is a **focal site's OPEX spike** (not a new site opening). For every neighbor within 20km, how does their OPEX and retail wash count change around the spike? No age filter — all neighbors are already established sites.

In [33]:
def analyze_incumbent_response_to_spikes(data, spikes, dist_df,
                                          pre_months=6, post_months=12, min_pre_obs=3):
    """
    For each OPEX spike at a focal site, measure how neighbors within 20km
    respond in OPEX and retail washes. Normalized by each neighbor's own
    pre-spike baseline. All neighbors qualify — no age filter needed.
    """
    rd = data.copy()
    rd['report_date'] = pd.to_datetime(rd['report_date'])
    rd['true_opex']   = rd['cogs'] + rd['expenses']

    rd_by_site = {sk: grp.sort_values('report_date') for sk, grp in rd.groupby('site_key')}

    nbr_lookup = {}
    for focal, grp in dist_df.groupby('focal_key'):
        nbr_lookup[focal] = list(zip(grp['neighbor_key'], grp['distance_km']))

    records = []
    isolated = 0
    for _, spike in spikes.iterrows():
        focal_key  = spike['site_key']
        spike_date = spike['report_date']
        neighbors  = nbr_lookup.get(focal_key, [])
        if not neighbors:
            isolated += 1
            continue

        for nbr_key, dist_km in neighbors:
            if nbr_key not in rd_by_site:
                continue
            nbr_ts = rd_by_site[nbr_key].copy()
            nbr_ts['mfs'] = (
                (nbr_ts['report_date'].dt.year  - spike_date.year)  * 12 +
                (nbr_ts['report_date'].dt.month - spike_date.month)
            )
            window = nbr_ts[(nbr_ts['mfs'] >= -pre_months) & (nbr_ts['mfs'] <= post_months)]
            pre    = window[window['mfs'] < 0]
            if len(pre) < min_pre_obs:
                continue

            baseline_opex = pre['true_opex'].mean()
            baseline_ret  = pre['ret_wash_count'].mean()
            if pd.isna(baseline_opex) or baseline_opex <= 0:
                continue

            for _, row in window.iterrows():
                records.append({
                    'focal_key':        focal_key,
                    'spike_date':       spike_date,
                    'neighbor_key':     nbr_key,
                    'distance_km':      dist_km,
                    'months_from_spike': int(row['mfs']),
                    'opex_norm':        row['true_opex'] / baseline_opex,
                    'ret_wash_norm':    (row['ret_wash_count'] / max(baseline_ret, 1)
                                        if pd.notna(row['ret_wash_count']) else np.nan),
                })

    spike_resp_df = pd.DataFrame(records)
    if len(spike_resp_df) > 0:
        n_pairs = spike_resp_df.groupby(['focal_key', 'spike_date', 'neighbor_key']).ngroups
        print(f"Spike events with no neighbors:     {isolated}")
        print(f"Neighbor-spike pairs:               {n_pairs}")
    else:
        print("No neighbor-spike pairs found.")
    return spike_resp_df

spike_resp_df = analyze_incumbent_response_to_spikes(data, spikes, dist_df)


Spike events with no neighbors:     89
Neighbor-spike pairs:               414


In [34]:
if len(spike_resp_df) == 0:
    print("No spike-incumbent pairs — cannot plot.")
else:
    INC_COLOR = '#E65100'

    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        subplot_titles=[
            'Neighbor OPEX (normalized) — do neighbors increase spend when a focal site runs a promotion?',
            'Neighbor Retail Wash Count (normalized) — do neighbors lose retail volume?',
        ],
        vertical_spacing=0.10,
    )

    for i, col in enumerate(['opex_norm', 'ret_wash_norm'], start=1):
        agg = (
            spike_resp_df.groupby('months_from_spike')[col]
            .agg(median='median',
                 q25=lambda s: s.quantile(0.25),
                 q75=lambda s: s.quantile(0.75),
                 n='count')
            .reset_index()
        )
        agg = agg[agg['n'] >= 5]
        x   = agg['months_from_spike']

        fig.add_trace(go.Scatter(
            x=list(x) + list(x[::-1]),
            y=list(agg['q75']) + list(agg['q25'][::-1]),
            fill='toself', fillcolor=INC_COLOR, opacity=0.12,
            line=dict(width=0), showlegend=False, hoverinfo='skip',
        ), row=i, col=1)
        fig.add_trace(go.Scatter(
            x=x, y=agg['median'],
            mode='lines+markers',
            line=dict(color=INC_COLOR, width=2.5), marker=dict(size=6),
            showlegend=False,
            hovertemplate='M%{x}<br>Median: %{y:.2f}× baseline<extra></extra>',
        ), row=i, col=1)

        fig.add_hline(y=1.0, line=dict(color='#333', dash='dash', width=1), row=i, col=1)
        fig.add_vline(x=0,   line=dict(color='#C62828', dash='dot', width=1.5), row=i, col=1)
        fig.update_yaxes(title_text='× baseline', tickformat='.2f', gridcolor='#EEE', row=i, col=1)
        fig.update_xaxes(showgrid=False, showline=True, linecolor='#CCC', row=i, col=1)

    n_pairs = spike_resp_df.groupby(['focal_key', 'spike_date', 'neighbor_key']).ngroups
    fig.update_xaxes(title_text='Months from OPEX spike', row=2, col=1)
    fig.update_layout(
        title=dict(
            text='<b>Neighbor Response to Focal Site Promotional Spike</b>'
                 f'<br><sub>{n_pairs} neighbor-spike pairs · '
                 'red dotted = spike month · dashed = pre-spike baseline · '
                 'shaded = 25–75th percentile</sub>',
            x=0.02, xanchor='left',
        ),
        plot_bgcolor='white',
        height=620,
        margin=dict(l=70, r=40, t=120, b=60),
    )
    fig.show()


## Section 6 — Promotions: Market Gain vs. Membership Churn
Two forces are always at play after a promotional OPEX spike:
- **Market Gain**: the site captures a larger share of washes within its 20km market
- **Churn**: new members acquired during the promo may not stick (mem_purchase_count falls back)

For each spike, we classify the outcome and compare trajectories for successful vs. unsuccessful operators.

In [35]:
def analyze_promotion_market_dynamics(data, spikes, dist_df,
                                       pre_months=6, post_months=12, min_pre_obs=3):
    """
    Event = OPEX spike. For each spike, track per month:
      - mem_purchase_count (new member acquisitions — promotion drives this up; falling back = churn)
      - mem_wash_count     (ongoing washes by all members — sticky if promo members retained)
      - market_share       (% of total washes within 20km including self)

    Classify each spike into:
      - 'Gained & Retained': market share at M+3 > pre-spike AND mem_purchase_count at M+6 >= 80% baseline
      - 'Gained & Churned' : market share at M+3 > pre-spike BUT  mem_purchase_count at M+6 <  80% baseline
      - 'No Gain'          : market share at M+3 <= pre-spike

    Returns promo_df with one row per (spike, month_offset), plus a summary table.
    """
    rd = data.copy()
    rd["report_date"]  = pd.to_datetime(rd["report_date"])
    rd["true_opex"]    = rd["cogs"] + rd["expenses"]
    rd["total_washes"] = rd["mem_wash_count"].fillna(0) + rd["ret_wash_count"].fillna(0)

    washes_pivot = rd.pivot_table(
        index="report_date", columns="site_key",
        values="total_washes", aggfunc="sum"
    )

    nbr_lookup = {}
    for focal, grp in dist_df.groupby("focal_key"):
        nbr_lookup[focal] = list(zip(grp["neighbor_key"], grp["distance_km"]))

    market_sites = {
        focal: [focal] + [n for n, _ in nbrs]
        for focal, nbrs in nbr_lookup.items()
    }

    rd_by_site = {sk: grp.sort_values("report_date").reset_index(drop=True)
                  for sk, grp in rd.groupby("site_key")}

    COLS = ["mem_purchase_count", "mem_wash_count", "ret_wash_count", "total_washes", "total_income"]

    records = []
    skipped = 0

    for _, spike in spikes.iterrows():
        site_key   = spike["site_key"]
        spike_date = pd.to_datetime(spike["report_date"])
        if site_key not in rd_by_site:
            skipped += 1
            continue

        ts = rd_by_site[site_key].copy()
        ts["mfs"] = (
            (ts["report_date"].dt.year  - spike_date.year)  * 12 +
            (ts["report_date"].dt.month - spike_date.month)
        )
        window = ts[(ts["mfs"] >= -pre_months) & (ts["mfs"] <= post_months)]
        pre    = window[window["mfs"] < 0]
        if len(pre) < min_pre_obs:
            skipped += 1
            continue

        baselines = {c: max(pre[c].mean(), 1e-6) for c in COLS}

        all_mkt = market_sites.get(site_key, [site_key])

        for _, row in window.iterrows():
            mfs = int(row["mfs"])
            dt  = row["report_date"]

            # market share
            if dt in washes_pivot.index:
                row_w    = washes_pivot.loc[dt]
                mkt_tot  = row_w.reindex(all_mkt).sum(min_count=1)
                ne_w     = row_w.get(site_key, np.nan)
                mkt_share = (ne_w / mkt_tot * 100
                             if pd.notna(mkt_tot) and mkt_tot > 0 else np.nan)
            else:
                mkt_share = np.nan

            rec = {
                "site_key":          site_key,
                "spike_date":        spike_date,
                "months_from_spike": mfs,
                "market_share":      mkt_share,
            }
            for c in COLS:
                rec[f"{c}_norm"] = row[c] / baselines[c] if pd.notna(row[c]) else np.nan
                rec[c]           = row[c]
            records.append(rec)

    promo_df = pd.DataFrame(records)
    if len(promo_df) == 0:
        print("No data.")
        return promo_df

    # ── Classify each spike ───────────────────────────────────────────────────
    def classify_spike(grp):
        pre_ms   = grp[grp["months_from_spike"] < 0]["market_share"].mean()
        post3_ms = grp[grp["months_from_spike"].between(1, 3)]["market_share"].mean()
        post6_mpc = grp[grp["months_from_spike"].between(4, 6)]["mem_purchase_count_norm"].mean()

        if pd.isna(pre_ms) or pd.isna(post3_ms):
            return "No Gain"
        if post3_ms > pre_ms:
            if pd.notna(post6_mpc) and post6_mpc >= 0.80:
                return "Gained & Retained"
            return "Gained & Churned"
        return "No Gain"

    labels = (
        promo_df.groupby(["site_key", "spike_date"])
        .apply(classify_spike, include_groups=False)
        .reset_index()
        .rename(columns={0: "outcome"})
    )
    promo_df = promo_df.merge(labels, on=["site_key", "spike_date"], how="left")

    n_spikes = promo_df.groupby(["site_key", "spike_date"]).ngroups
    summary  = promo_df.groupby(["site_key", "spike_date"])["outcome"].first().value_counts()

    print(f"Spikes analyzed    : {n_spikes}  (skipped {skipped})")
    print()
    print("── Outcome distribution ─────────────────────────────")
    for outcome, cnt in summary.items():
        print(f"  {outcome:<22}: {cnt:>4}  ({cnt/n_spikes*100:.1f}%)")

    # Delta table: median post-spike change for each outcome
    post = promo_df[promo_df["months_from_spike"].between(1, 6)]
    print()
    print("── Median post-spike delta (M+1 to M+6) ────────────")
    print(f"  {'Outcome':<22}  {'mkt_share_Δpp':>13}  {'mem_purch_norm':>14}  {'mem_wash_norm':>13}")
    print("  " + "-"*65)
    for outcome in ["Gained & Retained", "Gained & Churned", "No Gain"]:
        sub = post[post["outcome"] == outcome]
        if len(sub) == 0:
            continue
        pre_sub = promo_df[(promo_df["outcome"] == outcome) & (promo_df["months_from_spike"] < 0)]
        ms_pre  = pre_sub["market_share"].median()
        ms_post = sub["market_share"].median()
        ms_delta = (ms_post - ms_pre) if pd.notna(ms_pre) and pd.notna(ms_post) else np.nan
        mpc = sub["mem_purchase_count_norm"].median()
        mwc = sub["mem_wash_count_norm"].median()
        print(f"  {outcome:<22}  {ms_delta:>+12.1f}pp  {mpc:>13.2f}x  {mwc:>12.2f}x")

    return promo_df

promo_df = analyze_promotion_market_dynamics(data, spikes, dist_df)


Spikes analyzed    : 253  (skipped 2)

── Outcome distribution ─────────────────────────────
  No Gain               :  194  (76.7%)
  Gained & Retained     :   52  (20.6%)
  Gained & Churned      :    7  (2.8%)

── Median post-spike delta (M+1 to M+6) ────────────
  Outcome                 mkt_share_Δpp  mem_purch_norm  mem_wash_norm
  -----------------------------------------------------------------
  Gained & Retained               +6.4pp           1.34x          1.33x
  Gained & Churned                +9.1pp           3.20x          3.17x
  No Gain                         +0.0pp           1.14x          1.18x


In [36]:
if len(promo_df) == 0:
    print("No promo data to plot.")
else:
    OUTCOME_STYLES = {
        "Gained & Retained": {"color": "#2E7D32", "dash": "solid"},
        "Gained & Churned":  {"color": "#E65100", "dash": "dash"},
        "No Gain":           {"color": "#C62828", "dash": "dot"},
    }

    PANELS = [
        ("mem_purchase_count_norm", ""),
        ("mem_wash_count_norm",     "Member Wash Count (normalized)"),
        ("market_share",            "Market Share within 20km (%)"),
    ]

    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        subplot_titles=[p[1] for p in PANELS],
        vertical_spacing=0.08,
    )

    for row_idx, (col, label) in enumerate(PANELS, start=1):
        for outcome, style in OUTCOME_STYLES.items():
            sub = promo_df[promo_df["outcome"] == outcome]
            if len(sub) == 0:
                continue
            agg = (
                sub.groupby("months_from_spike")[col]
                .agg(median="median",
                     q25=lambda s: s.quantile(0.25),
                     q75=lambda s: s.quantile(0.75),
                     n="count")
                .reset_index()
            )
            agg = agg[agg["n"] >= 3]
            x   = agg["months_from_spike"]
            n_events = sub.groupby(["site_key", "spike_date"]).ngroups

            fig.add_trace(go.Scatter(
                x=list(x) + list(x[::-1]),
                y=list(agg["q75"]) + list(agg["q25"][::-1]),
                fill="toself", fillcolor=style["color"], opacity=0.08,
                line=dict(width=0), showlegend=False, hoverinfo="skip",
            ), row=row_idx, col=1)
            fig.add_trace(go.Scatter(
                x=x, y=agg["median"],
                mode="lines+markers",
                name=f"{outcome} (n={n_events})" if row_idx == 1 else None,
                legendgroup=outcome, showlegend=(row_idx == 1),
                line=dict(color=style["color"], width=2.5, dash=style["dash"]),
                marker=dict(size=5),
                hovertemplate=f"M%{{x}}<br>{outcome}: %{{y:.2f}}<extra></extra>",
            ), row=row_idx, col=1)

        # Reference line
        ref_y = 1.0 if col != "market_share" else None
        if ref_y is not None:
            fig.add_hline(y=ref_y, line=dict(color="#999", dash="dash", width=1),
                          row=row_idx, col=1)
        fig.add_vline(x=0, line=dict(color="#333", dash="dot", width=1.5),
                      row=row_idx, col=1)

        y_fmt   = ".1f" if col == "market_share" else ".2f"
        y_title = label.split("(")[0].strip()
        y_suf   = "%" if col == "market_share" else "×"
        fig.update_yaxes(title_text=y_title, tickformat=y_fmt,
                         ticksuffix=y_suf, gridcolor="#EEE", row=row_idx, col=1)
        fig.update_xaxes(showgrid=False, showline=True, linecolor="#CCC",
                         row=row_idx, col=1)

    n_total = promo_df.groupby(["site_key", "spike_date"]).ngroups
    fig.update_xaxes(title_text="Months from OPEX Spike", row=3, col=1)
    fig.update_layout(
        title=dict(
            text=(
                "<b>Promotion Outcomes: Market Gain vs. Membership Churn</b>"
                f"<br><sub>{n_total} spike events · "
                "green = gained market share & retained members · "
                "orange = gained share but members churned · "
                "red = no market share gain · "
                "shaded = 25–75th pct · dotted = spike month</sub>"
            ),
            x=0.02, xanchor="left",
        ),
        legend=dict(orientation="h", x=0.02, y=1.03, xanchor="left"),
        plot_bgcolor="white",
        height=820,
        margin=dict(l=90, r=40, t=140, b=60),
    )
    fig.show()


In [37]:
if len(promo_df) == 0:
    print("No promo data for Sankey.")
else:
    # One row per spike event → its outcome
    spike_outcomes = (
        promo_df.groupby(["site_key", "spike_date"])["outcome"]
        .first()
    )
    n_total        = len(spike_outcomes)
    n_ret          = (spike_outcomes == "Gained & Retained").sum()
    n_churn        = (spike_outcomes == "Gained & Churned").sum()
    n_no_gain      = (spike_outcomes == "No Gain").sum()
    n_gained       = n_ret + n_churn

    def pct(n): return f"{n/n_total*100:.1f}%"

    # ── Nodes ─────────────────────────────────────────────────────────────────
    # 0  All Spikes
    # 1  Market Share Gained
    # 2  No Market Share Gain
    # 3  Gained & Retained
    # 4  Gained & Churned

    node_labels = [
        f"All Spike Events<br>{n_total} promotions",
        f"Market Share Gained<br>{n_gained} spikes · {pct(n_gained)}",
        f"No Market Share Gain<br>{n_no_gain} spikes · {pct(n_no_gain)}",
        f"Gained & Retained<br>{n_ret} spikes · {pct(n_ret)}<br><i>members stuck</i>",
        f"Gained & Churned<br>{n_churn} spikes · {pct(n_churn)}<br><i>members fell back</i>",
    ]

    node_colors = [
        "#455A64",   # 0 All — slate
        "#1565C0",   # 1 Gained — blue
        "#C62828",   # 2 No Gain — red
        "#2E7D32",   # 3 Retained — green
        "#E65100",   # 4 Churned — orange
    ]

    # ── Links ─────────────────────────────────────────────────────────────────
    link_sources = [0,        0,          1,       1      ]
    link_targets = [1,        2,          3,       4      ]
    link_values  = [n_gained, n_no_gain,  n_ret,   n_churn]
    link_colors  = [
        "rgba(21,101,192,0.35)",    # all → gained
        "rgba(198,40,40,0.25)",     # all → no gain
        "rgba(46,125,50,0.40)",     # gained → retained
        "rgba(230,81,0,0.35)",      # gained → churned
    ]
    link_labels = [
        f"{pct(n_gained)} of promotions gained market share",
        f"{pct(n_no_gain)} of promotions did not move market share",
        f"{pct(n_ret)} retained members post-promo",
        f"{pct(n_churn)} saw member acquisition fall back",
    ]

    fig = go.Figure(go.Sankey(
        arrangement="snap",
        node=dict(
            pad=28,
            thickness=24,
            line=dict(color="white", width=0.5),
            label=node_labels,
            color=node_colors,
            hovertemplate="%{label}<extra></extra>",
        ),
        link=dict(
            source=link_sources,
            target=link_targets,
            value=link_values,
            color=link_colors,
            label=link_labels,
            hovertemplate="%{label}<extra></extra>",
        ),
    ))

    fig.update_layout(
        title=dict(
            text=(
                "<b>Promotion Outcome Funnel</b>"
                f"<br><sub>{n_total} OPEX spike events · "
                "does a promotion capture market share, and do those members stay?</sub>"
            ),
            x=0.02, xanchor="left",
        ),
        font=dict(size=13, family="Arial"),
        height=480,
        margin=dict(l=20, r=20, t=100, b=20),
        paper_bgcolor="white",
    )
    fig.show()
